# Visual Asset Auditing System - Test Bench

This notebook implements the **Test Bench** tier of the Visual Asset Auditing System. It demonstrates:

1. Connecting to AlloyDB and GCS.
2. Running a Hybrid Search (pgvector + FTS).
3. Detecting the drop-off point and selecting the top 60 candidates (High Confidence + Borderline).
4. Running Gemini 3.5 Flash online inference to audit the selected assets.
5. Creating and updating the `audit_results` table in AlloyDB.

*Transcribed from IMG_9422.jpeg. Additional source screenshots will be added below in the order received.*

In [ ]:
# Install required libraries
!pip install -q google-genai google-cloud-alloydb-connector google-cloud-storage pgvector asyncpg kneed pandas numpy pillow opencv-python-headless nest-asyncio sqlalchemy pydantic "protobuf<5.0.0dev"


## Package Installation

This cell installs all necessary external libraries (Google GenAI, Cloud Vision, AI Platform, pgvector, asyncio) to equip the notebook environment.

In [ ]:
# Authenticate with Google Cloud
from google.colab import auth
auth.authenticate_user()

import google.genai as genai
from google.genai import types

print("Google GenAI SDK imported successfully.")


## Google Cloud Authentication

This cell handles authentication with Google Cloud using Colab auth utilities, sets up API project client.

In [ ]:
# AlloyDB Connection Setup using SQLAlchemy Pool + AsyncConnector
import asyncio
import asyncpg
from typing import Tuple
from sqlalchemy.ext.asyncio import create_async_engine, AsyncEngine
from google.cloud.alloydb.connector import IPTypes, AsyncConnector

_engine_cache = {}
_connector_cache = {}

async def get_alloydb_connection(reuse: bool = True) -> Tuple[AsyncEngine, AsyncConnector]:
    """Establishes and pools AlloyDB connections using SQLAlchemy and the AsyncConnector, with automatic timeout diagnostics."""
    global _engine_cache
    global _connector_cache

    if reuse and 'default' in _engine_cache:
        return _engine_cache['default'], _connector_cache['default']

    # Use lazy refresh for serverless/Colab environments
    connector = AsyncConnector(refresh_strategy="lazy")

    async def getconn():
        # Handle case where user pasted the full resource path or just the instance ID
        instance_uri = ALLOYDB_INSTANCE
        if not instance_uri.startswith("projects/"):
            instance_uri = f"projects/{PROJECT_ID}/locations/{REGION}/clusters/{ALLOYDB_CLUSTER}/instances/{ALLOYDB_INSTANCE}"

        try:
            # Enforce 10-second connection timeout to prevent hanging loop CancelledErrors
            conn = await asyncio.wait_for(
                connector.connect(
                    instance_uri,
                    "asyncpg",
                    user=DB_USER,
                    password=DB_PASSWORD,
                    db=DB_NAME,
                    enable_iam_auth=False, # Set to True if using IAM auth
                    ip_type=IPTypes.PUBLIC # Adjust to PUBLIC or PRIVATE
                ),
                timeout=10.0
            )
            return conn
        except asyncio.TimeoutError:
            raise ConnectionError(
                f"AlloyDB connection timed out (10s) to {instance_uri}. "
                "Ensure your client IP is authorized in the AlloyDB Public IP console, "
                "or check your VPC network access if running internally."
            )
        except Exception as e:
            raise ConnectionError(f"Failed to connect to AlloyDB: {e}")

    engine = create_async_engine(
        "postgresql+asyncpg://",
        async_creator=getconn,
        echo=False,
        pool_size=10,
        max_overflow=20,
        pool_pre_ping=True # Force SQLAlchemy to health-check connections
    )

    if reuse:
        _engine_cache['default'] = engine
        _connector_cache['default'] = connector

    return engine, connector


In [ ]:
# # Run the setup
# import nest_asyncio
# nest_asyncio.apply()
# try:
#     asyncio.run(setup_database())
# except Exception as e:
#     print(f"Skipping execution: Database connection not configured yet ({e})")


In [ ]:
# Database Connection Verification (Read-Only Check – No DDL or Schema Changes)
import asyncio
import nest_asyncio

async def verify_database_connection():
    """Verifies connection to AlloyDB and confirms the visual_assets table is reachable without making any DDL changes."""
    try:
        engine, _ = await get_alloydb_connection()
        async with engine.connect() as conn:
            raw_conn = await conn.get_raw_connection()
            db = raw_conn.driver_connection
            count = await db.fetchval(f"SELECT COUNT(*) FROM {DB_SCHEMA}.visual_assets;")
            print(f"Connected to AlloyDB successfully. Schema '{DB_SCHEMA}.visual_assets' is ready ({count} assets found).")
    except Exception as e:
        print(f"Database connection status: {e}")

# Run connection check
nest_asyncio.apply()
try:
    asyncio.run(verify_database_connection())
except Exception as e:
    print(f"Skipping execution: Database connection not configured yet ({e})")


In [ ]:
# 1. Audit Config Generator & Embedding Generation
import json
import asyncio
from concurrent.futures import ThreadPoolExecutor
from typing import Optional, List, Dict, Tuple
from pydantic import BaseModel, Field

# Define Pydantic model for the structured Audit Context.
class AuditContextModel(BaseModel):
    audit_goal: str = Field(description="Refined, precise version of the user's goal.")
    image_description: Optional[str] = Field(None, description="Description of the reference image if provided, else null.")
    reference_is_composite_canvas: bool = Field(description="True if the reference image is a composite layout, webpage screenshot, hero banner, or real-world photograph containing the logo/asset. False if the reference image represents an isolated, standalone logo on a plain background.")
    inclusion_criteria: List[str] = Field(description="List of 3-7 specific, testable criteria an image MUST meet to be relevant.")
    exclusion_criteria: List[str] = Field(description="List of 2-5 criteria that EXCLUDE an image from relevance.")
    adjudication_logic: str = Field(description="A clear IF-THEN-ELSE statement defining PASS/FAIL conditions.")
    search_keywords: List[str] = Field(description="List of 5-10 single-word search terms (e.g. ['woman', 'female', 'portrait']) rather than multi-word phrases, to ensure broad keyword match capability in indexed assets.")
    vision_tag_filter: List[str] = Field(description="List of 2-5 keywords specifically mapped to Google Cloud Vision API tag vocabulary.")
    audit_instructions: str = Field(description="Detailed instructions to be passed to the auditing LLM describing the criteria.")
    extraction_schema: Dict[str, str] = Field(description="Dynamic key-value pairs representing additional boolean/integer/string properties to extract from the image to verify the audit criteria.")

def get_image_mime_type(path: str) -> str:
    """Helper to dynamically resolve visual asset MIME type based on file path extension."""
    lower_path = path.lower()
    if lower_path.endswith(".png"):
        return "image/png"
    elif lower_path.endswith(".webp"):
        return "image/webp"
    elif lower_path.endswith(".gif"):
        return "image/gif"
    return "image/jpeg"

def generate_audit_config(user_goal: str, reference_image_description: Optional[str] = None, available_tags: Optional[List[str]] = None) -> dict:
    """Translates a high-level user goal into a structured audit context with strict visual grounding and anti-hallucination guardrails."""
    image_context = ""
    if reference_image_description:
        image_context = f"\nReference Image Description: {reference_image_description}\n"

    active_tags = available_tags if available_tags else WEB_AUDIT_VISION_TAGS
    tags_pool_str = ", ".join([f"'{t}'" for t in active_tags])

    builder_prompt = f"""
You are a Lead Enterprise Visual & UI/UX Asset Auditor building a high-precision audit configuration for executive leadership.
Your task is to expand a user's audit goal into an airtight, zero-mistake structured audit context.

User Goal: {user_goal}
{image_context}

BRAND COMPLIANCE SIGNATURES INFERENCE:
Analyze the User Goal and the Reference Image Description (if provided) to extract:
1. The target brand, UI component, or visual subject under audit (e.g. Google Pay, YouTube, a specific Favicon, a cookies banner, or Google Workspace logos).
2. What constitutes the COMPLIANT (active, modern, approved) visual design, layout, typography, or shape.
3. What constitutes the NON-COMPLIANT (legacy, outdated, spoofed, or incorrect) visual design, layout, typography, or shape.
Ground all inclusion and exclusion criteria strictly in these inferred compliance signatures.

STEP 1: DYNAMIC BRAND & INTENT EXTRACTION
Identify the target brand/visual subject under audit based on the brand compliance signatures inference. Restrict all criteria, search keywords, and instructions strictly to this extracted subject.

STRICT VISUAL GROUNDING & ANTI-HALLUCINATION (CRITICAL):
- You MUST base all inclusion/exclusion criteria and visual descriptions strictly and exclusively on what is physically visible inside the provided reference image.
- Do NOT assume, extrapolate, or hallucinate the presence of brand names, wordmarks, UI elements, button texts, or logos that are cropped out or missing from the reference image.
- If a button, text, or logo is not visible in the reference image, do NOT include it as a mandatory requirement (inclusion criteria).

STEP 2: COMPLIANCE STATE ALIGNMENT & TEMPLATE ROLE ANALYSIS
Analyze the role of the reference image template (if provided) using your inferred compliance signatures.
- **State Conflict (Negation/Comparative Match)**: If the reference image represents the *Active/Compliant/New* standard, but the user wants to find *outdated/old* assets.
  * The template role is **Negative / Comparative Match**.
  * The exclusion criteria MUST exclude the reference image's compliant visual signatures.
  * The inclusion criteria MUST target older legacy styles of that same brand.
- **State Alignment (Positive Template Match)**: If the reference image represents the *Outdated/Legacy/Old* standard, and the user wants to find *outdated/old* assets.
  * The template role is **Positive Template Match**.
  * The inclusion criteria MUST require matching the reference image's legacy visual signatures.
  * The exclusion criteria MUST explicitly exclude the modern, compliant standard.
- **General Discovery Match**: If the user wants to find *all* assets of the brand regardless of state.
  * The inclusion criteria should pass the reference style AND other iterations of the brand.

STEP 3: REFERENCE COMPOSITE CANVAS DETECTION
Evaluate the description of the reference image:
- Set `reference_is_composite_canvas` to True if it describes a composite scene, real-world photograph, or webpage screenshot containing the logo/asset.
- Set `reference_is_composite_canvas` to False only if it represents an isolated, standalone logo on a plain background.

STEP 4: STRICT BRAND EXCLUSION & ANTI-SPOOFING GUARDRAIL (CRITICAL)
If the target visual subject is a specific sub-brand or product logo:
- You MUST explicitly include in the `exclusion_criteria` a rule to exclude the generic corporate master logo unless it is explicitly accompanied by the sub-brand.
- ANTI-SPOOFING: Add explicit rules to reject look-alike misspellings or related but incorrect sub-brands.

STEP 5: COLOR & CANVAS CONTEXT INDEPENDENCE (CRITICAL)
Unless the user's goal explicitly specifies a color constraint, you MUST explicitly write in the `audit_instructions` and `adjudication_logic` that color is not a match determinant.

CRITICAL RULE FOR vision_tag_filter:
You MUST ONLY select tags from the following list that exist in our database:
[{tags_pool_str}]

CRITICAL RULE FOR search_keywords (CRITICAL):
- The search_keywords MUST be a list of single-word search terms rather than multi-word phrases.

CRITICAL MANDATES FOR OPTICAL RESOLUTION & CLARITY AUDITS (CRITICAL):
1. **Strict Adjudication Logic**: Write a crystal-clear IF-THEN-ELSE statement in `adjudication_logic`.
   - Recognize Cropped/Blurred Targets: State explicitly that if the target visual asset is cropped, low-resolution, or blurred, but you can still identify its core structural signatures, it remains eligible.
   - Flag Resolution Status: In such cases, you must mark `optical_resolution_sufficient = False` in the extraction schema.
   - Only evaluate `matches_criteria = False` if the image is so extremely degraded, pixelated, or tiny (sub-pixel) that it is mathematically impossible to distinguish it from a generic shape.
2. **Diagnostic Extraction Schema**: In `extraction_schema`, define:
   - `detected_asset_style`: string (Exact description of what is seen on canvas)
   - `is_outdated_or_noncompliant`: boolean
   - `is_embedded_in_composite_hero`: boolean
   - `composite_location_notes`: string
   - `optical_resolution_sufficient`: boolean
"""

    # Passing the Pydantic model directly to the SDK
    response = client.models.generate_content(
        model=GEMINI_ORCHESTRATOR_MODEL,
        contents=builder_prompt,
        config=types.GenerateContentConfig(
            response_mime_type="application/json",
            response_schema=AuditContextModel,
            temperature=0.0
        )
    )
    return json.loads(response.text)

def build_fused_query_text(context_dict: dict, is_negative: bool = False) -> str:
    """Combines all context fields into a single rich text representation for embedding."""
    if is_negative:
        return f"Exclude images that: {'; '.join(context_dict.get('exclusion_criteria', []))}. Specifically exclude spoofed misspellings, lookalike brands, and other products."

    parts = [
        f"Audit goal: {context_dict.get('audit_goal')}",
        f"Include images that: {'; '.join(context_dict.get('inclusion_criteria', []))}",
        "IMPORTANT FOR EMBEDDING SIMILARITY: Ignore foreground and background color differences. Focus purely on shape, text layout, logo structural design, and semantic meaning."
    ]
    if context_dict.get("image_description"):
        parts.append(f"Reference image: {context_dict.get('image_description')}")
    return " | ".join(parts)

async def embed_audit_context(context_dict: dict, reference_image_path: Optional[str] = None) -> Tuple[List[float], List[float]]:
    """Generates a POSITIVE and NEGATIVE multimodal embedding for Contrastive Retrieval."""
    pos_text = build_fused_query_text(context_dict, is_negative=False)
    neg_text = build_fused_query_text(context_dict, is_negative=True)

    pos_contents = []
    if reference_image_path:
        mime = get_image_mime_type(reference_image_path)
        if reference_image_path.startswith("gs://"):
            pos_contents.append(types.Part.from_uri(file_uri=reference_image_path, mime_type=mime))
        else:
            with open(reference_image_path, "rb") as f:
                pos_contents.append(types.Part.from_bytes(data=f.read(), mime_type=mime))

    pos_contents.append(pos_text[:1000])
    neg_contents = [neg_text[:1000]]

    def _embed(contents):
        return client.models.embed_content(
            model=EMBEDDING_MODEL,
            contents=contents,
            config=types.EmbedContentConfig(output_dimensionality=768)
        ).embeddings[0].values

    loop = asyncio.get_running_loop()
    with ThreadPoolExecutor(max_workers=2) as executor:
        pos_future = loop.run_in_executor(executor, _embed, pos_contents)
        neg_future = loop.run_in_executor(executor, _embed, neg_contents)
        pos_vec, neg_vec = await asyncio.gather(pos_future, neg_future)

    return pos_vec, neg_vec


In [ ]:
# 2. Parallel Hybrid Search with RRF & Drop-Off Detection
from typing import List, Tuple, Optional
import numpy as np
import pandas as pd
from kneed import KneeLocator
from pgvector.asyncpg import register_vector
import asyncio
from concurrent.futures import ThreadPoolExecutor

async def run_semantic_reranking_and_filter(df_high: pd.DataFrame, df_edge: pd.DataFrame, audit_context: dict, max_workers: int = 25) -> Tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    """Applies LLM Cross-Encoder semantic scoring to the pre-segmented candidates, with visual vector safeguards to protect recall against description gaps."""
    candidates_df = pd.concat([df_high, df_edge], ignore_index=True)
    if candidates_df.empty:
        return pd.DataFrame(), pd.DataFrame(), pd.DataFrame()

    candidates = candidates_df.to_dict(orient="records")
    audit_goal = audit_context.get("audit_goal", "")

    def _score_candidate(row):
        # 1. Visual Vector Safeguard Check (Absolute visual distance check)
        # Cosine distance < 0.28 means high visual similarity (approx >0.72 similarity).
        # If this is triggered, we flag it to bypass Cross-Encoder checks.
        vec_dist = row.get("vector_distance", 1.0)
        if vec_dist < 0.28:
            return {
                **row,
                "cross_encoder_score": 100, # Max score to force-promote
                "relevance_score": row.get("relevance_score", 0.0) * 2.0, # Visual match boost
                "vector_safeguard_triggered": True
            }

        desc = row.get("gemini_description", "")
        tags = ", ".join(row.get("vision_tags") or [])
        filename = row.get("asset_filename", "")

        prompt = f"""You are a rapid relevance scoring engine.
Audit Goal: {audit_goal}
Candidate Description: {desc}
Candidate Tags: {tags}
Candidate Filename: {filename}

Score the candidate's textual relevance on a scale of 0 to 100 using this calibrated rubric:
- 80-100 (High): Direct matches to the target subject, clear presence of target visual elements, or standalone brand logos requested.
- 30-79 (Borderline): Contextual matches, related terms/brands, composite graphics containing the target, or candidate descriptions with some visual layout complexity.
- 0-29 (Low): Completely unrelated elements, different subjects (e.g. illustrations when looking for photos, different products, or unrelated graphics).

Output a single integer from 0 to 100 representing the score. Output NOTHING ELSE.
"""
        try:
            resp = client.models.generate_content(
                model=GEMINI_CROSS_ENCODER_MODEL,
                contents=prompt,
                config=types.GenerateContentConfig(temperature=0.0)
            )
            score = int(resp.text.strip())
        except:
            score = 50 # Default neutral fallback

        ce_mult = max(0.1, score / 50.0)
        new_score = row.get("relevance_score", 0.0) * ce_mult

        return {
            **row,
            "cross_encoder_score": score,
            "relevance_score": new_score,
            "vector_safeguard_triggered": False
        }

    print(f"[Cross-Encoder] Semantic reranking of {len(candidates)} candidate assets...")
    loop = asyncio.get_running_loop()
    with ThreadPoolExecutor(max_workers=max_workers) as executor:
        tasks = [loop.run_in_executor(executor, _score_candidate, item) for item in candidates]
        reranked = await asyncio.gather(*tasks)

    high_list = []
    edge_list = []
    low_list = []

    for item in reranked:
        if item.get("vector_safeguard_triggered", False):
            filename = item.get("asset_filename") or item["gcs_raw_path"].split("/")[-1]
            dist = item.get("vector_distance", 0)
            print(f"🛡️ Vector Safeguard triggered: Force-promoted {filename[:40]} due to high visual similarity (distance: {dist:.4f})")
            high_list.append(item)
            continue

        score = item["cross_encoder_score"]
        if score >= 75:
            high_list.append(item)
        elif score >= 30:
            edge_list.append(item)
        else:
            low_list.append(item)

    df_high_final = pd.DataFrame(high_list).sort_values(by="relevance_score", ascending=False).reset_index(drop=True) if high_list else pd.DataFrame()
    df_edge_final = pd.DataFrame(edge_list).sort_values(by="relevance_score", ascending=False).reset_index(drop=True) if edge_list else pd.DataFrame()
    df_low_final = pd.DataFrame(low_list).sort_values(by="relevance_score", ascending=False).reset_index(drop=True) if low_list else pd.DataFrame()

    return df_high_final, df_edge_final, df_low_final

def compute_weighted_rrf_rerank(candidates: list, audit_context: dict) -> list:
    """Zero-latency in-memory multi-factor reranker. Executes in local CPU RAM (< 0.5ms) without external API overhead."""
    tag_filter = [t.lower().strip() for t in audit_context.get("vision_tag_filter", []) if t]
    search_keywords = [k.lower().strip() for k in audit_context.get("search_keywords", []) if k]
    reranked = []

    for item in candidates:
        row = item["data"]
        base_score = item["score"]
        multiplier = 1.0

        # 1. Vision tag exact hit boost (2.0x per matching tag up to 8x)
        tags = [str(t).lower() for t in (row.get("vision_tags") or [])]
        tag_hits = sum(1 for t in tags if any(ft in t or t in ft for ft in tag_filter))
        if tag_hits > 0:
            multiplier *= (2.0 ** min(tag_hits, 3))

        # 2. Keyword exact hit in filename or description boost (1.5x per matching keyword up to 2.25x)
        desc = str(row.get("gemini_description") or "").lower()
        fname = str(row.get("asset_filename") or "").lower()
        kw_hits = sum(1 for kw in search_keywords if kw in desc or kw in fname)
        if kw_hits > 0:
            multiplier *= min(1.5 ** kw_hits, 2.25)

        reranked.append({
            "data": row,
            "score": base_score * multiplier,
            "base_rrf_score": base_score,
            "rerank_multiplier": multiplier,
            "vector_distance": item.get("vector_distance", 1.0)
        })

    reranked.sort(key=lambda x: x["score"], reverse=True)
    return reranked

async def run_hybrid_search(scope_config: dict, audit_context: dict, reference_image_path: Optional[str] = None, limit: int = 10000) -> List[dict]:
    """Performs parallel 3-Arm Vector + Keyword + Tag Boosting search with RRF fusion, local reranking, and deduplication (No silent cliff)."""
    pos_vec, neg_vec = await embed_audit_context(audit_context, reference_image_path)

    engine, _ = await get_alloydb_connection()

    search_keywords = audit_context.get("search_keywords", [])
    keyword_query_str = " OR ".join(search_keywords) if search_keywords else ""

    tag_filter = audit_context.get("vision_tag_filter", [])
    tag_filter = tag_filter if tag_filter else []

    # Run the 3 database query arms concurrently using pooled SQLAlchemy connections.
    async def run_vector():
        async with engine.connect() as conn:
            raw_conn = await conn.get_raw_connection()
            db = raw_conn.driver_connection
            await register_vector(db)
            rows = await db.fetch(
                f"""
                SELECT asset_id, gcs_raw_path, format, vision_tags, gemini_description, asset_filename, page_url, content_hash,
                       (embedding <=> $1::vector) as vector_distance
                FROM {DB_SCHEMA}.visual_assets
                ORDER BY $1::vector <=> embedding - (0.3 * (embedding <=> $2::vector)) ASC
                LIMIT $3
                """,
                pos_vec, neg_vec, limit
            )
            return [dict(r) for r in rows]

    async def run_fts():
        if not keyword_query_str:
            return []
        async with engine.connect() as conn:
            raw_conn = await conn.get_raw_connection()
            db = raw_conn.driver_connection
            rows = await db.fetch(
                f"""
                SELECT asset_id, gcs_raw_path, format, vision_tags, gemini_description, asset_filename, page_url, content_hash
                FROM {DB_SCHEMA}.visual_assets
                WHERE to_tsvector('english', gemini_description) @@ websearch_to_tsquery('english', $1)
                LIMIT $2
                """,
                keyword_query_str, limit
            )
            return [dict(r) for r in rows]

    async def run_tags():
        if not tag_filter:
            return []
        async with engine.connect() as conn:
            raw_conn = await conn.get_raw_connection()
            db = raw_conn.driver_connection
            rows = await db.fetch(
                f"""
                SELECT asset_id, gcs_raw_path, format, vision_tags, gemini_description, asset_filename, page_url, content_hash
                FROM {DB_SCHEMA}.visual_assets
                WHERE EXISTS (
                    SELECT 1 FROM unnest(vision_tags) tag
                    WHERE EXISTS (
                        SELECT 1 FROM unnest($1::text[]) filter_tag
                        WHERE tag ILIKE '%' || filter_tag || '%'
                    )
                )
                LIMIT $2
                """,
                tag_filter, limit
            )
            return [dict(r) for r in rows]

    vector_task = run_vector()
    fts_task = run_fts()
    tag_task = run_tags()

    vector_results, fts_results, tag_results = await asyncio.gather(vector_task, fts_task, tag_task)

    promoted_ids = set()
    for r in fts_results[:100]:
        promoted_ids.add(str(r["asset_id"]))
    for r in tag_results[:100]:
        promoted_ids.add(str(r["asset_id"]))

    # 3-Arm RRF Fusion (k=60)
    k = 60
    results_map = {}

    # Pre-populate with vector distances
    vector_distance_map = {str(r["asset_id"]): r["vector_distance"] for r in vector_results}

    def upsert_ranks(results_list, weight=1.0):
        for rank, row in enumerate(results_list):
            img_id = str(row["asset_id"])
            if img_id not in results_map:
                results_map[img_id] = {"data": row, "score": 0.0, "vector_distance": vector_distance_map.get(img_id, 1.0)}
            results_map[img_id]["score"] += weight / (k + rank + 1)

    upsert_ranks(vector_results, weight=0.60)
    if fts_results:
        upsert_ranks(fts_results, weight=0.25)
    if tag_results:
        upsert_ranks(tag_results, weight=0.15)

    fused = list(results_map.values())

    # Zero-Latency In-Memory Reranking (Provides a smooth score curve for Kneedle)
    reranked_fused = compute_weighted_rrf_rerank(fused, audit_context)

    # Deduplication
    seen_identifiers = set()
    deduplicated = []
    for item in reranked_fused:
        row = item["data"]
        img_id = str(row["asset_id"])
        img_identifier = row.get("content_hash") or row.get("gcs_raw_path")
        if img_identifier not in seen_identifiers:
            seen_identifiers.add(img_identifier)
            is_promoted = img_id in promoted_ids
            deduplicated.append({
                **row,
                "relevance_score": item["score"],
                "base_rrf_score": item.get("base_rrf_score", 0),
                "rerank_multiplier": item.get("rerank_multiplier", 1),
                "promoted_by_keyword_or_tag": is_promoted,
                "vector_distance": item.get("vector_distance", 1.0)
            })

    return deduplicated[:limit]

def detect_dropoff_flawless(df: pd.DataFrame, sensitivity: float = 1.0) -> Tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    """Applies Kneedle curvature + rolling volatility to segment candidates into High, Edge, and Low."""
    if df is None or df.empty:
        return pd.DataFrame(), pd.DataFrame(), pd.DataFrame()

    df_sorted = df.sort_values(by="relevance_score", ascending=False).reset_index(drop=True)
    y = df_sorted["relevance_score"].values
    x = np.arange(len(y))

    y_min, y_max = y.min(), y.max()
    if y_max == y_min:
        return df_sorted.iloc[:int(len(y)*0.3)], df_sorted.iloc[int(len(y)*0.3):int(len(y)*0.6)], df_sorted.iloc[int(len(y)*0.6):]

    y_norm = (y - y_min) / (y_max - y_min + 1e-9)
    x_norm = x / (len(x) - 1)

    coords = np.column_stack((x_norm, y_norm))
    line_start, line_end = coords[0], coords[-1]
    line_vec = line_end - line_start
    line_vec_norm = line_vec / np.sqrt(np.sum(line_vec**2))
    vec_from_start = coords - line_start
    scalar_proj = np.dot(vec_from_start, line_vec_norm)
    proj_on_line = np.outer(scalar_proj, line_vec_norm)
    dist_to_line = np.sqrt(np.sum((coords - proj_on_line)**2, axis=1))

    idx1 = np.argmax(dist_to_line)

    window = max(3, int(len(y) * 0.05))
    rolling_std = pd.Series(y_norm).rolling(window=window, center=True).std().fillna(0).values
    noise_threshold = np.mean(rolling_std) * (0.6 / sensitivity)

    idx2 = len(y) - 1
    for i in range(idx1 + 2, len(rolling_std)):
        if rolling_std[i] < noise_threshold:
            idx2 = i
            break

    min_borderline_width = max(10, int((len(y) - idx1) * 0.25))
    if (idx2 - idx1) < min_borderline_width:
        idx2 = min(len(y) - 1, idx1 + min_borderline_width)

    idx1 = max(10, idx1)

    high_df = df_sorted.iloc[:idx1 + 1].copy()
    edge_df = df_sorted.iloc[idx1 + 1: idx2 + 1].copy()
    low_df = df_sorted.iloc[idx2 + 1:].copy()

    # Visual Vector Safeguard: Check if any candidate has extremely high similarity (distance < 0.28)
    # even if it is currently classified in Low_df (or has been discarded).
    # Force rescue these to protect visual recall.
    if "vector_distance" in low_df.columns:
        rescued_vec = low_df[low_df["vector_distance"] < 0.28].copy()
        if not rescued_vec.empty:
            edge_df = pd.concat([edge_df, rescued_vec], ignore_index=True)
            low_df = low_df[low_df["vector_distance"] >= 0.28].copy()
            print(f"🛡️ Vector Safeguard triggered in Kneedle: Force-rescued {len(rescued_vec)} candidate(s) from Low to Borderline based on high visual similarity.")

    if "promoted_by_keyword_or_tag" in low_df.columns:
        rescued = low_df[low_df["promoted_by_keyword_or_tag"] == True].copy()
        if not rescued.empty:
            edge_df = pd.concat([edge_df, rescued], ignore_index=True)
            low_df = low_df[low_df["promoted_by_keyword_or_tag"] != True].copy()
            print(f"🛡️ Safeguard triggered: Promoted {len(rescued)} composite/diluted candidates from Low to Borderline tier based on exact keyword/tag match.")

    return high_df, edge_df, low_df


## Retrieval, Reranking, and Drop-off Segmentation Engine

This cell defines the core search engine:

- `run_hybrid_search`: Combines Vector, Full-Text, and Tag search arms into a fused RRF list, applying zero-latency keyword boosts.
- `detect_dropoff_flawless`: Curvature-based Kneedle algorithm that truncates irrelevant tail results.

In [ ]:
# 3. Hydration & Parallel LLM Audit Inference
import asyncio
from concurrent.futures import ThreadPoolExecutor
import json
import pandas as pd
from typing import List, Tuple, Optional
from google.genai import types
from google.cloud import storage
from pydantic import create_model, Field
import time
from PIL import Image
import io

def process_transparency(image_bytes: bytes, default_bg: Tuple[int, int, int] = (30, 30, 30)) -> bytes:
    """Detects alpha transparency in an image and composites it onto a solid dark background to ensure light elements/text remain visible."""
    try:
        img = Image.open(io.BytesIO(image_bytes))
        if img.mode in ('RGBA', 'LA') or (img.mode == 'P' and 'transparency' in img.info):
            img = img.convert('RGBA')
            # Create a solid dark grey background
            bg = Image.new("RGBA", img.size, default_bg + (255,))
            alpha_composite = Image.alpha_composite(bg, img)
            final_img = alpha_composite.convert("RGB")
            out_bytes = io.BytesIO()
            final_img.save(out_bytes, format="PNG")
            return out_bytes.getvalue()
    except Exception as e:
        print(f"Warning: Failed to preprocess image transparency: {e}")
    return image_bytes

def enforce_zero_false_positives_rules(df: pd.DataFrame) -> pd.DataFrame:
    """Enforces zero-false-positive criteria. Demotes matches if confidence is below 75%."""
    if df.empty:
        return df

    def guardrail_check(row):
        matches = bool(row.get("matches_criteria", False))
        conf = int(row.get("match_confidence", 100))
        rationale = str(row.get("visual_analysis_step_by_step", "")) + " " + str(row.get("match_rationale", ""))
        if matches and conf < 75:
            row["matches_criteria"] = False
            row["match_rationale"] = f"[GUARDRAIL DEMOTION: Conf {conf}% < 75%] {rationale}"
        return row

    return df.apply(guardrail_check, axis=1)

async def run_llm_audit_single(asset_data: dict, audit_config: dict, _executor=None, reference_image_part: Optional[types.Part] = None) -> dict:
    """Evaluates a single image asset against dynamic JSON schema using Pydantic, supporting side-by-side reference comparisons and transparency blending."""
    audit_instructions = audit_config.get("audit_instructions", "")
    extraction_schema = audit_config.get("extraction_schema", {})
    inclusion_criteria = audit_config.get("inclusion_criteria", [])
    exclusion_criteria = audit_config.get("exclusion_criteria", [])
    adjudication_logic = audit_config.get("adjudication_logic", "")
    is_composite = bool(audit_config.get("reference_is_composite_canvas", False))

    inclusion_str = "\n".join([f"- {c}" for c in inclusion_criteria]) if inclusion_criteria else "- None specified"
    exclusion_str = "\n".join([f"- {c}" for c in exclusion_criteria]) if exclusion_criteria else "- None specified"

    fields = {
        "visual_analysis_step_by_step": (
            str,
            Field(description="CHAIN OF THOUGHT: Write a comprehensive, step-by-step visual analysis of the image BEFORE making any conclusions. Describe exact shapes, typography, brand marks, colors, and layout.")
        ),
        "matches_criteria": (
            bool,
            Field(description="Strict final evaluation: True ONLY if the asset matches the target criteria and passes adjudication_logic (even inside a composite hero banner). False otherwise.")
        ),
        "match_confidence": (
            int,
            Field(description="Match Confidence percentage (0-100%). You must assign < 75 if there is any doubt or visual occlusion.")
        ),
        "match_rationale": (
            str,
            Field(description="A concise final executive rationale explaining exactly why matches_criteria evaluated to True or False based on the visual_analysis_step_by_step.")
        )
    }

    for field_name, field_info in extraction_schema.items():
        if field_name in ["matches_criteria", "match_confidence", "match_rationale", "visual_analysis_step_by_step"]:
            continue

        t = str
        desc = f"Extracted value for {field_name}"

        if isinstance(field_info, dict):
            ftype = field_info.get("field_type", "string").lower()
            desc = field_info.get("description", desc)
        else:
            ftype = str(field_info).lower()

        if ftype == "boolean":
            t = bool
        elif ftype == "integer":
            t = int
        elif ftype == "number":
            t = float

        fields[field_name] = (t, Field(description=desc))

    DynamicAuditModel = create_model("DynamicAuditModel", **fields)

    reference_instructions = ""
    if reference_image_part:
        if is_composite:
            reference_instructions = f"""
You are given two images:
- The FIRST image (image_0) is the Reference Image (the template provided by the user).
- The SECOND image (image_1) is the Candidate Image (the asset under audit).

REFERENCE TEMPLATE DETAILS:
- Audit Goal: {audit_config.get('audit_goal')}
- Reference Target Description: {audit_config.get('image_description', 'No description')}

CANVAS ISOLATION DIRECTIVE (CRITICAL):
The reference image (image_0) is a **Composite Canvas** (a complex real-world photograph/screenshot containing the logo).
Do NOT expect the candidate image under audit (image_1) to contain the hands, terminals, backgrounds, or full layout seen in image_0.
Instead, look at the target logo/brand style (e.g. logos or wordmark shown on the phone screen) inside image_0.
Verify if the candidate image (image_1) contains that target logo style. Ignore all other background visual noise in image_0.
"""
        else:
            reference_instructions = f"""
You are given two images:
- The FIRST image (image_0) is the Reference Image (the template provided by the user).
- The SECOND image (image_1) is the Candidate Image (the asset under audit).

REFERENCE TEMPLATE DETAILS:
- Audit Goal: {audit_config.get('audit_goal')}
- Reference Image Description: {audit_config.get('image_description', 'No description')}

CANVAS ISOLATION DIRECTIVE:
Since the Reference Image (image_0) is a standalone logo, compare the candidate image (image_1) side-by-side against image_0.
"""

    # Updated prompt template: Objective, clean, and safe from prompt override classifications
    audit_prompt = f"""
Evaluate this image against the specified audit goal and criteria with high precision.

{reference_instructions}

[Color Independence Rule]:
Unless the Inclusion Criteria explicitly mention a required color, you must ignore any color differences between the reference image and the candidate image.

[Exact Typography Rule]:
Pay strict attention to typography and spelling (e.g. 'Google Play' is NOT 'Google Pay', 'Ads' is NOT 'AdWords'). Reject any assets that contain lookalike or misspelled branding unless the inclusion criteria explicitly permit them.

AUDIT SCOPE & EVALUATION RULES:

[Inclusion Criteria – Asset MUST fulfill these to pass]:
{inclusion_str}

[Exclusion Criteria – If asset triggers any of these, it MUST fail]:
{exclusion_str}

[Strict Adjudication Rule]:
{adjudication_logic}

[General Audit Instructions]:
{audit_instructions}

EXECUTION STEPS:
1. Provide a detailed, step-by-step visual analysis of the image in the `visual_analysis_step_by_step` field. Scan the entire canvas to inspect all visual details, shapes, and text.
2. Extract all diagnostic visual features requested in the schema, including whether the target is embedded inside a composite hero graphic (`is_embedded_in_composite_hero`).
3. Apply the Strict Adjudication Rule against your extracted visual findings.
4. Set `matches_criteria` to true if the asset satisfies the inclusion criteria without triggering any exclusion criteria with 100% certainty.
5. Provide a clear justification in `match_rationale` explaining your decision based on your step-by-step analysis.
"""

    gcs_path = asset_data["gcs_raw_path"]

    # Resolve bytes locally or from GCS to handle transparency rendering
    def _download_and_preprocess():
        if gcs_path.startswith("gs://"):
            bucket_name = gcs_path.split("/")[2]
            blob_name = "/".join(gcs_path.split("/")[3:])
            client_storage = storage.Client(project=PROJECT_ID)
            bucket = client_storage.bucket(bucket_name)
            blob = bucket.blob(blob_name)
            img_bytes = blob.download_as_bytes()
        else:
            with open(gcs_path, "rb") as f:
                img_bytes = f.read()
        return process_transparency(img_bytes)

    loop = asyncio.get_running_loop()
    try:
        # Download and composite image on background thread
        processed_bytes = await loop.run_in_executor(_executor, _download_and_preprocess)
        image_part = types.Part.from_bytes(data=processed_bytes, mime_type="image/png")

        contents = []
        if reference_image_part:
            contents.append(reference_image_part)

        contents.append(image_part)
        contents.append(audit_prompt)

        def _call_gemini():
            return client.models.generate_content(
                model=GEMINI_INFERENCE_MODEL,
                contents=contents,
                config=types.GenerateContentConfig(
                    response_mime_type="application/json",
                    response_schema=DynamicAuditModel,
                    temperature=0.0
                )
            )

        response = await loop.run_in_executor(_executor, _call_gemini)
        extracted_data = json.loads(response.text)
    except Exception as e:
        extracted_data = {
            "matches_criteria": False,
            "match_confidence": 0,
            "match_rationale": f"Audit evaluation failed due to error: {str(e)}",
            "error": str(e)
        }

    return {**asset_data, **extracted_data}

async def run_llm_inference_on_dropoff_results(df_high: pd.DataFrame, df_edge: pd.DataFrame, audit_config: dict, max_workers: int = 15, reference_image_path: Optional[str] = None) -> pd.DataFrame:
    """Runs parallel multi-threaded LLM inference on candidate subsets, supporting side-by-side template matches, GCS/local transparency preprocessing, and caching."""
    candidates_df = pd.concat([df_high, df_edge], ignore_index=True)
    if candidates_df.empty:
        return pd.DataFrame()

    candidates = candidates_df.to_dict(orient="records")

    print(f"Starting parallel LLM audit inference on {len(candidates)} candidates (Max concurrency: {max_workers})...")

    # Helper to download and preprocess the reference image once
    def _get_reference_part():
        if reference_image_path.startswith("gs://"):
            bucket_name = reference_image_path.split("/")[2]
            blob_name = "/".join(reference_image_path.split("/")[3:])
            client_storage = storage.Client(project=PROJECT_ID)
            bucket = client_storage.bucket(bucket_name)
            blob = bucket.blob(blob_name)
            ref_bytes = blob.download_as_bytes()
        else:
            with open(reference_image_path, "rb") as f:
                ref_bytes = f.read()
        processed_ref = process_transparency(ref_bytes)
        return types.Part.from_bytes(data=processed_ref, mime_type="image/png")

    t0 = time.time()
    loop = asyncio.get_running_loop()
    with ThreadPoolExecutor(max_workers=max_workers) as executor:
        reference_image_part = None
        if reference_image_path:
            reference_image_part = await loop.run_in_executor(executor, _get_reference_part)

        tasks = [run_llm_audit_single(asset, audit_config, _executor=executor, reference_image_part=reference_image_part) for asset in candidates]
        results = await asyncio.gather(*tasks)

    duration = time.time() - t0
    print(f" Processing assets... [{len(candidates)}/{len(candidates)}] completed")
    print(f" Inference completed. Enforcing Zero-FP policies and sorting scoreboard...")

    results_df = pd.DataFrame(results)
    results_df = enforce_zero_false_positives_rules(results_df)

    results_df = results_df.sort_values(by="relevance_score", ascending=False).reset_index(drop=True)

    print(f"\n=== VERIFIED AUDIT SCOREBOARD (Sorted by Search Similarity) ===")
    for i, r in results_df.iterrows():
        status_label = "PASS" if r.get("matches_criteria", False) else "FAIL"
        filename = r.get("asset_filename") or r.get("gcs_raw_path", "").split("/")[-1]
        sim = r.get("relevance_score", 0.0)
        conf = r.get("match_confidence", 100)
        print(f"[{i+1}/{len(results_df)}] {status_label} | {filename[:40]} | Sim: {sim:.4f} | Conf: {conf}%")

    return results_df


In [ ]:
# 4. Calibration Summary & Audit Results Saving E2E Loop
async def save_audit_results_to_db(session_id: str, results_df: pd.DataFrame):
    """Saves the final audited results into AlloyDB (Bypassed by default in the interactive runner)."""
    if results_df.empty:
        return

    engine, _ = await get_alloydb_connection()
    async with engine.connect() as conn:
        raw_conn = await conn.get_raw_connection()
        db = raw_conn.driver_connection
        records = results_df.to_dict("records")
        for r in records:
            verdict = "PASS" if r.get("matches_criteria") is True else "FAIL"

            # Extract criteria_checks from JSON response or construct it
            criteria_checks = {k: v for k, v in r.items() if k not in ["asset_id", "gcs_raw_path", "matches_criteria", "error", "asset_filename", "page_url"]}

            await db.execute(
                f"""
                INSERT INTO {DB_SCHEMA}.audit_results (
                    session_id, asset_id, overall_verdict, adjudication_result,
                    criteria_checks, rationale, confidence_band
                ) VALUES ($1, $2, $3, $4, $5, $6, $7)
                """,
                session_id,
                r.get("asset_id"),
                verdict,
                r.get("matches_criteria", False),
                json.dumps(criteria_checks),
                r.get("match_rationale", "Completed"),
                "high" if r.get("match_confidence", 0) > 95 else ("borderline" if r.get("match_confidence", 0) >= 70 else "below_threshold")
            )
    print("Audit results saved to database.")

async def generate_ai_audit_summary(results_df: pd.DataFrame, audit_config: dict) -> str:
    """Generates an AI-powered executive summary of the visual asset audit results."""
    import json
    import numpy as np
    if results_df is None or results_df.empty:
        return "No audit results available to generate a summary."

    # Select columns to pass to the LLM, avoiding internal or verbose vector columns
    exclude_cols = {"num_chunks", "max_relevance_score", "gcs_raw_path", "gcs_processed_path", "embedding", "embedding_at"}
    cols_to_include = [col for col in results_df.columns if col not in exclude_cols]

    # Convert to records safely, handling NumPy arrays, lists, and floats without boolean truth value ambiguity
    clean_df = results_df[cols_to_include].copy()
    for col in clean_df.columns:
        def safe_clean(val):
            if val is None:
                return None
            if isinstance(val, (np.ndarray, pd.Series)):
                return val.tolist() if val.size > 0 else None
            if isinstance(val, float) and pd.isna(val):
                return None
            return val
        clean_df[col] = clean_df[col].apply(safe_clean)

    records = clean_df.to_dict(orient="records")
    formatted_results = json.dumps(records, indent=2)

    audit_instructions = audit_config.get("audit_instructions", "No specific audit context provided.")

    # Compute basic stats to seed in the prompt
    total_audited = len(results_df)
    matches_col = "matches_criteria" if "matches_criteria" in results_df.columns else None
    if matches_col:
        # Convert to boolean safely, handling string representation if any
        matches_true = results_df[matches_col].apply(lambda x: str(x).lower() in ("true", "1", "yes")).sum()
    else:
        matches_true = "N/A"

    summary_prompt = f"""
You are a Lead Visual Asset Auditor.
Your task is to write a visually engaging, highly structured, and extremely concise summary of a visual asset audit Test Bench calibration run.

STRICT RULES FOR FORMATTING & SECTIONS:
1. You MUST only include the following exact three sections in the output:
   - Objective & Scope (Preview Subset) (within the top [!NOTE] block)
   - Calibration Statistics (as a numbered list)
   - Configuration Calibration Insights (as a single, brief narrative paragraph of 3-4 sentences detailing the visual rules performance)
2. DO NOT include any other sections.
3. DO NOT pass any definitive verdicts of success.

AUDIT CONTEXT (Visual Evaluation Criteria):
{audit_instructions}

TEST BENCH STATS:
- Total images audited: {total_audited}
- Total images matching criteria (True): {matches_true}

TEST BENCH FINDINGS (JSON):
{formatted_results}
"""

    response = client.models.generate_content(
        model=GEMINI_ORCHESTRATOR_MODEL,
        contents=summary_prompt,
        config=types.GenerateContentConfig(temperature=0.0)
    )
    return response.text


## Telemetry Summaries & Calibration Database Saving

This cell defines functions to generate final executive AI summaries (`generate_ai_audit_summary`) and save audit session details into the run calibration tables in AlloyDB for telemetry history.

In [ ]:
# 5. E2E Execution Helpers (Interactive Split Stages)
import base64
import os
import time
import json
import pandas as pd
from google.cloud import storage

def get_gcs_image_base64(gcs_path: str) -> str:
    """Downloads image from GCS or local file and returns its base64 data URI for inline HTML rendering."""
    try:
        image_bytes = b""
        mime_type = "image/png"

        # Detect format
        lower_path = gcs_path.lower()
        if lower_path.endswith(".jpg") or lower_path.endswith(".jpeg"):
            mime_type = "image/jpeg"
        elif lower_path.endswith(".webp"):
            mime_type = "image/webp"
        elif lower_path.endswith(".gif"):
            mime_type = "image/gif"

        if gcs_path.startswith("gs://"):
            parts = gcs_path.replace("gs://", "").split("/", 1)
            bucket_name = parts[0]
            blob_name = parts[1]

            # Authenticated download using python client
            storage_client = storage.Client()
            bucket = storage_client.bucket(bucket_name)
            blob = bucket.blob(blob_name)
            image_bytes = blob.download_as_bytes()
        elif os.path.exists(gcs_path):
            with open(gcs_path, "rb") as f:
                image_bytes = f.read()
        else:
            return ""

        encoded = base64.b64encode(image_bytes).decode("utf-8")
        return f"data:{mime_type};base64,{encoded}"
    except Exception as e:
        return ""

async def generate_audit_config_only(user_goal: str, reference_image_path: Optional[str] = None) -> dict:
    """Stage 1: Analyzes reference image (Forensic) and generates structured Audit Configuration."""
    reference_image_description = None
    if reference_image_path:
        print("0. Performing forensic executive analysis on reference image...")
        ref_mime = get_image_mime_type(reference_image_path)
        if reference_image_path.startswith("gs://"):
            image_part = types.Part.from_uri(file_uri=reference_image_path, mime_type=ref_mime)
        else:
            with open(reference_image_path, "rb") as f:
                image_part = types.Part.from_bytes(data=f.read(), mime_type=ref_mime)

        forensic_analysis_prompt = """You are a Lead Executive Visual & UI/UX Inspector. Perform an exhaustive, forensic-level breakdown of this uploaded reference image for enterprise audit configuration.
Provide a comprehensive, structured analysis starting with Brand Hierarchy:
1. Target Brand & Scope: What exact brand or product is shown? (e.g. specifically Google Pay / G Pay, or Google Workspace). Do not mix in separate products.
2. Primary Asset Hierarchy & Category: Classify whether this image is a Standalone Brand Logo, a UI Component (Payment Button, Cookie Modal), or a Composite Canvas (Hero Banner, Screenshot, Collage).
3. Exact Visual Signatures: Detail exact wordmarks, typography, overlapping geometric shapes (e.g. interlocking loops vs text wordmark), border radius, padding, and layout structure.
4. Color & Contrast Styling: Detail exact colors, contrast levels, and shadow/elevation effects.
5. Compliance & Style Classification: Classify whether this image represents an active/compliant style or an outdated/deprecated legacy pattern.
-Guideline for Google Pay (G Pay) compliance status:*
- The CURRENT COMPLIANT standard is the multi-colored interlocking loops design (four colored curved segments in blue, red, yellow, green forming a stylized double-loop G/Pay shape).
- Any logo showing the 'Google Pay' or 'G Pay' wordmark (where 'Google' is multi-colored and 'Pay' is grey/black/white) is an OUTDATED/LEGACY pattern."""

        resp = client.models.generate_content(
            model=GEMINI_ORCHESTRATOR_MODEL,
            contents=[image_part, forensic_analysis_prompt],
            config=types.GenerateContentConfig(temperature=0.0)
        )
        reference_image_description = resp.text
        print(f"Forensic Reference Image Analysis:\n{reference_image_description}\n{'='*50}")

    # Dynamically query all unique tags from AlloyDB visual_assets
    available_tags = []
    try:
        engine, _ = await get_alloydb_connection()
        async with engine.connect() as conn:
            raw_conn = await conn.get_raw_connection()
            db = raw_conn.driver_connection
            rows = await db.fetch(
                f"SELECT DISTINCT tag FROM {DB_SCHEMA}.visual_assets, unnest(vision_tags) tag"
            )
            available_tags = [r["tag"] for r in rows if r["tag"]]
            print(f"Loaded {len(available_tags)} unique tag vocabulary terms from database.")
    except Exception as e:
        print(f"Warning: Could not fetch unique tags from DB ({e}). Using static fallback.")

    print("1. Translating goal into Audit Configuration...")
    audit_config = generate_audit_config(user_goal, reference_image_description, available_tags)
    print("Generated Configuration:\n", json.dumps(audit_config, indent=2))
    return audit_config

async def run_full_test_bench_pipeline_execution(audit_config: dict, reference_image_path: Optional[str] = None, quick_mode: bool = False) -> pd.DataFrame:
    """Stage 2, 3, 3.5, 4, 5: Executes Retrieval, Semantic Reranking, Inference, and Calibration scorecards."""
    pipeline_start_time = time.time()
    telemetry = {}

    t0 = time.time()
    print("\n2. Running Parallel 3-Arm Hybrid Search with RRF...")
    search_results = await run_hybrid_search({}, audit_config, reference_image_path)
    telemetry["Stage 2 (3-Arm RRF Retrieval)"] = f"{time.time() - t0:.2f}s | {len(search_results)} candidates retrieved"
    print(f"Found {len(search_results)} candidates.")

    t0 = time.time()
    print("\n3. Running Drop-off Analysis (Kneedle + Volatility)...")
    df_results = pd.DataFrame(search_results)
    df_high, df_edge, df_low = detect_dropoff_flawless(df_results)
    telemetry["Stage 3 (Drop-off Segmentation)"] = f"{time.time() - t0:.2f}s | High: {len(df_high)}, Borderline: {len(df_edge)}, Low (Filtered): {len(df_low)}"
    print(f"Rough Candidates -> High: {len(df_high)} | Borderline: {len(df_edge)} | Low: {len(df_low)}")

    t0 = time.time()
    print("\n3.5. Running Semantic Reranking and Fine Filtering...")
    df_high_sem, df_edge_sem, df_low_sem = await run_semantic_reranking_and_filter(df_high, df_edge, audit_config)
    telemetry["Stage 3.5 (Semantic Reranking)"] = f"{time.time() - t0:.2f}s | High: {len(df_high_sem)}, Borderline: {len(df_edge_sem)}"
    print(f"Semantic Candidates -> High: {len(df_high_sem)} | Borderline: {len(df_edge_sem)} | Low (Discarded): {len(df_low_sem)}, Low (Demoted): {len(df_low_sem)}")

    # Save full results globally for evaluation recall calculation
    globals()["df_results_full"] = df_results

    # Apply Quick Mode vs Smart Scan selection
    if quick_mode:
        print("\n Quick Mode enabled: Selecting top 30 High confidence and top 30 Borderline for visual inference.")
        candidates_high = df_high_sem.head(30) if not df_high_sem.empty else pd.DataFrame()
        candidates_edge = df_edge_sem.head(30) if not df_edge_sem.empty else pd.DataFrame()
    else:
        print("\n Smart Scan enabled: Selecting all High confidence and all Borderline for full visual inference.")
        candidates_high = df_high_sem
        candidates_edge = df_edge_sem

    t0 = time.time()
    print("\n4. Running Parallel LLM Audit Inference...")
    results_df = await run_llm_inference_on_dropoff_results(candidates_high, candidates_edge, audit_config, reference_image_path=reference_image_path)
    inf_duration = time.time() - t0
    throughput = len(results_df) / inf_duration if inf_duration > 0 else 0
    telemetry["Stage 4 (Parallel Visual Inference)"] = f"{inf_duration:.2f}s | Throughput: {throughput:.2f} images/sec"
    print(f"LLM Results: {len(results_df)} assets audited.")

    t0 = time.time()
    print("\n5. Generating Calibration Summary...")
    summary = await generate_ai_audit_summary(results_df, audit_config)
    telemetry["Stage 5 (Executive Summary Generation)"] = f"{time.time() - t0:.2f}s"

    total_time = time.time() - pipeline_start_time

    # Compute Observability Stats
    error_count = results_df["error"].notna().sum() if (not results_df.empty and "error" in results_df.columns) else 0
    high_conf = (results_df["match_confidence"] >= 95).sum() if (not results_df.empty and "match_confidence" in results_df.columns) else 0
    borderline_conf = ((results_df["match_confidence"] >= 70) & (results_df["match_confidence"] < 95)).sum() if (not results_df.empty and "match_confidence" in results_df.columns) else 0
    low_conf = (results_df["match_confidence"] < 70).sum() if (not results_df.empty and "match_confidence" in results_df.columns) else 0

    try:
        from IPython.display import display, Markdown, HTML

        telemetry_md = f"""
### 📊 Enterprise Observability & Telemetry Scorecard
| Stage / Metric | Value / Duration | Status |
|:--- |:--- |:---|
| **Total Pipeline Execution Time** | **{total_time:.2f}s** | 🟢 Optimal Throughput |
| **Parallel Inference Speed** | **{throughput:.2f} images/sec** | Concurrency: 15 Workers |
| **Stage 2: 3-Arm RRF Retrieval** | {telemetry.get('Stage 2 (3-Arm RRF Retrieval)', 'N/A')} | HNSW + FTS + Tag Boost |
| **Stage 3: Kneedle Segmentation** | {telemetry.get('Stage 3 (Drop-off Segmentation)', 'N/A')} | Noise Tail Truncated |
| **Stage 3.5: Semantic Reranking** | {telemetry.get('Stage 3.5 (Semantic Reranking)', 'N/A')} | LLM Text Cross-Encoder |
| **Stage 4: Vision Inference** | {telemetry.get('Stage 4 (Parallel Visual Inference)', 'N/A')} | 0 False Positive Verdict Guarantee |
| **Confidence Band Distribution** | High (>95%): **{high_conf}** || Borderline (70-95%): **{borderline_conf}** || Low (<70%): **{low_conf}** || Error Count: **{error_count}** |
"""
        display(Markdown(telemetry_md))
        display(Markdown(summary))

        if not results_df.empty:
            display(Markdown("### Detailed Audit Results"))
            display_df = results_df.copy()
            display_df["Visual Preview"] = display_df["gcs_raw_path"].apply(
                lambda x: f'<img src="{get_gcs_image_base64(x)}" width="150" />' if get_gcs_image_base64(x) else '[No Preview]'
            )
            display_df["Page Link"] = display_df["page_url"].apply(
                lambda x: f'<a href="{x}" target="_blank">{x}</a>' if x else '[No Page Link]'
            )
            cols = ["Visual Preview", "matches_criteria", "relevance_score", "gcs_raw_path", "Page Link", "match_rationale"]
            col_labels = ["Visual Preview", "Matches Criteria", "Search Similarity", "GCS Path", "Page Location URL", "Rationale"]

            if "gemini_description" in display_df.columns:
                cols.append("gemini_description")
                col_labels.append("Image Description")

            display_df = display_df[cols]
            display_df.columns = col_labels

            display(HTML(display_df.to_html(escape=False, index=False)))
    except (ImportError, ModuleNotFoundError):
        print("\n=== TELEMETRY SCORECARD ===")
        for k, v in telemetry.items():
            print(f"  {k}: {v}")
        print(f"  Total Time: {total_time:.2f}s | Errors: {error_count}")
        print("\n=== CALIBRATION SUMMARY ===")
        print(summary)

    return results_df

# Bypassed original monolithic E2E pipeline name to map to partitioned runners
async def run_full_test_bench_pipeline(user_goal: str, reference_image_path: Optional[str] = None, quick_mode: bool = False) -> pd.DataFrame:
    """Wrapper to maintain backwards compatibility for existing cells."""
    config = await generate_audit_config_only(user_goal, reference_image_path)
    return await run_full_test_bench_pipeline_execution(config, reference_image_path, quick_mode=quick_mode)


## E2E Execution & Telemetry Helpers

This cell defines the core pipeline orchestrators: `generate_audit_config_only` (Stage 1 Config) and `run_full_test_bench_pipeline_execution` (Stage 2 E2E). It coordinates retrieval, segmentation, cross-encoder reranking, and visual LLM inference, while logging telemetry and formatted scorecard widgets.

## Recall & Precision Upgrade — Code Only

This optional override layer keeps the existing AlloyDB schema unchanged. It broadens candidate retrieval, ranks all search lanes, preserves canonical asset-to-page occurrence mappings, removes irreversible drop-off gating, and expands verified canonical matches back to every matching page occurrence.

In [ ]:
# 6. Recall & Precision Upgrade — self-contained, code only
# No AlloyDB schema changes, DDL, DML, or separate file upload is required.
import base64
import zlib

_optimizer_source = zlib.decompress(base64.b64decode(
    "eNrtfdty3DiW4Lu+Ap3TuyJdVOriS3mzOh0jy3KVtn2pld3T26vJoJlJpsQSk8wimbqM2xvztB+wsd+wH9ZfsucCkAAIplKy"
    "qyc2oiu6LSYJHBwAB+eGg4PBYHCazKIs212WySyt0iIXxVVSlmmcVGJelKK+SMS/pNUqysRhVSW1OFzFaZ3m5+KoyKLpcGvr"
    "40VaiUURr7JEwFOa10leAyCAeiviqI6mUZXsVLOLZBGJPFnVZZQNhTipRZlEcSWKHMpBM1vJTVoR5CtqL4ywvUrMimy1yCux"
    "qpJYTKmoyIs6mRbFpYjyGGACxtB0tQKcX716s/vq7Zvh1mAw2Nqal8VChOF8Va/KJAxFulgWZQ21AECESFZbW+pddZvP0kL9"
    "vIiqiyydqp/th0W6SOrbZVKpF2XCzQCeWTIjoKqdOJlHq6yO01mtyuSzVVnCAA0Zp6boxwscjZ+LIju+SWaruii5BjSFQyJL"
    "Hea3gXgF4AIYv6SMplkSiDcwbIF4v+RBD8TH1TJLmn7lq8XyFnon8qV6tYRRgxfwv2XMrfx88kY1cbKIzgEo/Xm/hPHZgl7A"
    "rFY1TGhYErWEDbWEDbV4eTUi1M6qugwQ04kvdl6Id0WejLYE/AczcsJgYBziZAcnPhAMcWeelhWO5TKLZskCBogoqRCRKFd5"
    "jkOg5hxIDqF9+pRXnz6JxQqqTRODKrYr+HqeFdMoqzz/0yegto/w2QAeJzC7MIJ1kt0SPOhSlZRXDKkhRUW/OHNAuLN6BKAN"
    "+gQcSqBsGCCiZ+rVUPWXUS3K9DyFqQnjslgW87kYi7w6G8RJDfSiXobzLLrOkqoaTMxK50lOeIaAwjw9l5WbtxGuR/ktxMbt"
    "+lm2CNN8ngDVzRJZG8bUfA81G0RgIIBoEQ8ChNN/ntRhihQRIvmHSP/eMqovRgImm6YZ/vIs439YKBAhNNasluE5rM6qrek3"
    "hcsEFkJOJUU657+4rPFhCORS1tV1Wl94A0Jgd+CLJIMJkT9/WSbngxbRZVnMqJ0yyqtlhH279Rjz6W2dAInSn0AtzXB6PuIF"
    "cwbkFgj1zwRw9x7vBYL/71MfGUKDOMzvYXYd3VaqCxFSQCZ+fvdjIKqCi1NX4mSWAS6xeHvy9hiXNPTwvEySIZKIAleXty1s"
    "/A97zetwWCyT3EuL4UsEefJe7xGgBgu5KlblLDHr43/8Hjqj1vMQSHsuh6eoEo8L+J2K/yReFitAHUg+hUXBJA+c4zYrgGnD"
    "itp/trcnljfYdyL+GP9m4voiBTkQXRVpDOvHAbZKcvwCCzer051Fch5hN8STP+4+/6OQLB/WPfL0W41fiuuivEzKYQfiIroJ"
    "K2BA0EdEqfMdSSq6kf0cVum/Jb540dTqjhiTZBUtlhmiOUbaj+q69CRjBIGpPg4km/SHbw7fHf2P9x+cwGTL9cVqMc1hiDxP"
    "NR40aPiB1qbv6oOEAnIWVkkuvMHpjy8PAYHBm0NYESCoPb3EeCwGPw+I9gb6UhhgXVkQ1n7h9/T/fBpBz2VBmASYjFo26Ttr"
    "TKPZ5XlJFCNpbZgn1w2WCJDGXl944jvhHTx9GvhukETjDbQoW15EwOZASFVpnXhtgwzdN9B0YIlMY7S2IUd3HXCKVb1c1VBc"
    "W46OGSOkq+gq8bhCgNrUIqrHA2APMCIFiOsFDMj4Y7lyLD/JUbjuEEjwKspWidZQcjNLlrU4pj+otgETgHdmD5clsDJvPvhz"
    "VKIIHYk50B+wIVhfOWKTAQKy+6jryUXeCIWR+Awgv1iDIFHTWFDLfsPzAiROndzUni6XnIqBITGoe8CdlVZzhsWQBxtN6zBx"
    "ULwBv8FWcS0M/OCO8ow1qCuzMqVxc1UbiMHwlyLNPUf9fJatSPEBAIBrGgGAswmtQPizMRwY183gdORkAxTGz6Nh83ny8BGX"
    "Nw8l8gz+OsyK66QE0mlnaVlkKcjFdVMUoAYMqkBDCqBYZTRvZrl2CnHWYcZ6SECnXJApUKBcVDjBA/qN0wD8T1Ij/qrRrGh+"
    "AY8Eo2LG0wxCfo5SZgmYgA6Nr+bRVQoNaaQKa5jGt20oA1kDLBBKF1lMf1Y16HcJPQO3wj9S2uDjeRnFqXyuYMComIRKjxHM"
    "Gny2e8YM2Bw+4sRRfushNjhJNFhkX8kX2qh0+yBBrgVg9LcFAVZJUWqD4OwVFlqV6okeSh6WKVAQD0aSUK9vkwzoCZ8WRV7M"
    "LsB6oAIg9+uEawBnHtgIpFUYp7joFqCT1ixYteFC+tX7Cr/XdlbrVWeBfDbXIMIbjASTWQi18iQqwxhUPiAnQBkXioYJ6Zae"
    "mucQtBUQN6ABUTkDR9ZCK9D9kTJDZp4dVtLTfcCo54tZ/5/IcgHTtwRd4baxzdFIgdbVUs/JaPlQIyM/EDOYIjZ5wO4D69SE"
    "t8rTX4FRzKK8yHFB4VOc4joArRgsKNEMDWh7AK+Y8ZoAtdpUvoBE6jJNAIdwuoqBq0GfUNx0mV2nIKrU4d4eqNXWaOFE3wVN"
    "LxOIZ2440fmdYNoifVCUnUd17wDnKhuI75/uuejrabcxmmj4NCUdKkOdmwWzUrl5cthahtk9UrNGijE6XqZRPbtIYmuS0Mpr"
    "tGjgfiPShveDnm7YxQPx3O+MCxai1kJU6TYBqZV2QyygWyEQdzpH0sM11ow3At/rBd5bMRAHezT4XUbs9XMeaVzu+d35ORQV"
    "aExZwFMEWhSIoSkoTuSlgeUHbHGnmhUlOhrYk5BK5xZ5roB9WQAbpHZ4pql3FbvGYFEimBms1EiADQw9EslimsTYcIqmZoYv"
    "sdxVVFmAp8kc0fhvq3R2Kd5iL6MlrGnkFGBczdMbgPF4D9Rv+OfXVVFHQ9dscPP3mgezSiD2D77hDJhtALsMabQBszkskDuR"
    "aisEYm/45PkmNHifhbO+diCeGB360mpj01WaxeEciSQEzlzesvKEDifkdOg87CpnIDZyUGdQMrFqBvL0dQQD11WtB4PBUZkg"
    "p4gEwSfHAug+AlTtCkUxSaLdqr4FnSq5QlULpml6q0w1w0sB06m37bIMBseo3MaSgYE1fxFBDwZAb4MflPKq924jjXiraYmV"
    "V+hvFwaI66gERsNFoP7nL616sERPElQ7M3CeD8ihLVBrBXunA3O7tTC2A7G97X8ZWHRzmvy6StG5I82nKj0HSY6+3bs7vbE5"
    "MdEnwAGnY9ZYlj11fghsIMljMAhPbdaTR9ltlVbWCJxtdwBvT3R7EJDhsWY0+pSddcgMjrAS0E6O+wpFjq4zUEt3kCesqh2l"
    "r4oaaRh9twFxEDXYRLSg5cDqBBGDXihQCqHdHFrQVfOO9e9CAhh2lcAAxBGsklWO3lhgmTjec+gkUA+55CLU1lFHgk8SDTl7"
    "Ra632Rhsf1UEgG2SHksPwMaZJnEU4UG30NCV3PJMNrOlx1U5+pEdsCXdvKFimkkGgFFpwHqCWq0vDI/qeTXa3bVnR6KNfvs7"
    "fJPklUSgaC9MB2sdkRKq9LFg97rOjPcfjsuyKPvRoQ+0USO90n0+cA815pDXNHHNdiiZoh1DCawBG1lr4zYu+Og6gtnv89G3"
    "zbtb9rvMTNnkDCAgnu456/oWNmc228Oe8OOWrl0CcUcxrzFy8kD3VrRdBXBWOfpdkMJRkwHzWN9L4Z0UDRYt850UlCBcPCQt"
    "Stz9I2Xn+iKR1uyuFOu8NGkLJ0liXFHDLgM562UeE5MeuFiz4Do+IuLJaDQ1dg3vERn7O6Xi2QSt5S06a2k4khh0GyjmBIc1"
    "M9QJUOJhF93MC9kUfqXBAfjpNEuGJljTyabEAXQvS6vau4cPasvyHCMHQGO6Nf3JCVyXHlReNO4hrD2whqOnJHIwfEOWuETB"
    "4UlWnxSTtWbO6rCkZUfnkJ7VD2cdXv4aRVeDiZMuup5DoxK5AjuVQIL/a/6atsJxCWG1QNJNSyJIN2p81ewTKei0Vbbq1dBF"
    "VK8Kya5B7ESSDGFFVbAWQNuH1QPDT66RDMYiR8puESDKArUOVOd5wuqHiNM58I/1dKaGMP4FOiYV16w4T2ebDGG3Uu8QipPX"
    "tGIk8XcWHvxwL522i5YY1tefYzA//nT8jvzdP4jjNx+OSWlK85Va/7hj28YaCLCcZ5dV71CxI93W+6zACWZkIwuZ+eCz5HDb"
    "aN6A/gT6QOMREWwrjZsytq9kezIKvmhOG+mqQZ3Cakb2hAi0A9bhniDIg35HM8+0LXPJCG0F7gamyj1FL2/GYkzDGdl1Ew5w"
    "kD8mhl1zjMjQhIK4YZ6trEzpPAFeInibCAqy9YOoDltzAkVFVRdSGKwWOWmBdZplVD6F9yTNMgyi2EmrH2CNSlNcBQhAjQbc"
    "NTCqGpYi7k/hPsthlhW3r14OdbRbtQp3xuWWPD0PWj1/lhH580f+oX1tECCbOZPFjt++PH716uTdj+Hb96+O32jll0XF0wV6"
    "8YgHVO6wnBmmhXOyDEKj/fmxMyJgvaqzpoW7lVK7D0qmcGzBz1B7iI6YcFWmHu4N4MPY1VQgGnzH+GRtPrp3CaUPDtU0p2J+"
    "d8eVrsOA3PuQm/SP20OiG0tYazrkAriBt8E/Gz1Gf6imSoC930BCmtkAiuGj4F3Os9F++BTgTtrVR8YOEbPnIE9iCNrqty2D"
    "agliG6mR1wftfWfVkHmUhNcVYVRqbK2gwCXpCJ+xegh6hOGYJ4mY0RGXPWIzgLdvYTgWuI2A7C6tb8ffP3tu+Z6cG6yqe8MG"
    "0+psbzJkn387gllRLNEg4fA1FMyhjJgK8ZNnksNVMgtoNuGhMWOaukC2oOAZyCCMIYYLpXmYyMg0D5l1IOctMMjM6tgGtXXS"
    "WrfvaWNvCyZs5eJ2WqaxNIfaflQzsFT79jm3OpvFSML9xTYXZ22dDDS4mnYOMEqFtj74Y0vgZnOmmDvVQ+S4j5oGwaZ4noC6"
    "B6pILQ7fvRNZBL+XuGlUxER0msA7qTk8h209UF+u0mKFYZaLJe09fPr0/vTV8al4+ZdPn9DRAXRYkV/m+iKdXYAxF7d+AbQB"
    "b8SqYh9STG3zFgUpn+lsldUoBYFrVqglT2FFsIM7yip4HeWXreh8/fHDbh2dCxl/ptzYp6ev2eVzmSRL5WBo/ZRojHCwEJok"
    "bjlbJucwxmCSA90ABaogOPOtLl1zMOtlBBsvEQ65q8MIxXk8RTLJOdhzMNlkibnUJuPXxo4Co9awSmrppXV4Pzv7/NzQGteC"
    "vvWMYxIS5WIcX3Tj0XPQ2OudXT1NXpAUUHVVBX3jrh1r3IfrlGz35tqCl8ntdVHGJH/QJkVHnw+aQ5kupU2qdofNISJTRQ6N"
    "gqEbzBRdZcMz8HtYk1ccIIt9AY0ECt6jUUvBeKDi0X6tLqKDp884KgHDmof8wrPg+cOL5CZOz5Oqhv4ZyhrjQVszhmewSjDo"
    "OVRB2mMxoBi+MAXWfj6rwjK6lnoXxz8Foh2XCookC7DDdAdzwEGAOGZJHmEcKW5Dgz6XBUokh9iLQcvRWiGgqBbYn7cBb20r"
    "ExPlZT+Ua9sjfyb+cHgzo+tmaWMJFrvQ1ZYvOILC4ilUglLDuASdqNQKd4oyaItDefHUESlWXFcNLvF0OE9qXfaZNuJg4Pzw"
    "4fjN8dFH8dmczC+Bs7D8z2ttoD+MX4jf749GjKUvDj+oiYgB/QjI516QDgxISoVsYDlBvT59/1Z8Bi69/epl+OHop+O3h2Bp"
    "D41IbWdFJetEX3ectd6cvD35KH7/2PkRRjnoVfBROjg/Sonh/qhz426J3vDBM1TEPaAQZlbwgLwKKWbiWj3zurrX0pHORcVU"
    "e53+momJ/7EZDpwCBl/tjCgY/v8Ha/NbrLg/n3z8if3iwFWByr3eBfKQtUlyC3lvfhnOYu+uolS8COtKspntKl0ss2SbGO4s"
    "qsPrytsW28EmcPC/o/eHb44/HB17Lu6+ve3fH5AtEh4GpCwjMFSLEMVtfu4Zggj65xPcTcD6m7V+nUyl3kGjS3SvDe7v9+9s"
    "jDgg6U04l72lH8z9iBR/Oj49/vr5/7pJf/hMf6vp9X3xz//80Cnz14nVRzw/crWvl0LNXItX0CmlC6XxOil0cD8pRH1yf2rV"
    "9t9OyJC+f38hg9O4qYD5h9h4uNjYSFhI4Efv//Tuo/doI5ZJS2AFgwOmRbNkjSV6eHp6+JezyWiEVAiGEvE+pn8osEkbzMmO"
    "//vJh48fNuuI1pl9A0XU/yQeiAbuF8NrMOE3RKVFhzdNm374sGr/eCy2/9O2+Otf5UcDuo/v4fNGcmgz+QEr7iJdIwEeLD6+"
    "lu1J0SPxEy/E3h3cUZX8zZgjkWLfl9+aNXJQJBq3D+GQtrX/D24pCfEV8IOTd/Dw/p3wdBeC32GP39a2ZPLWW4Qe/X5/PY3r"
    "xe8g8V4qtknhW5OsNEfxbSBVFn6ENcJPTMrGJK7bZTD8NoFmiAaavhB0F4i2XaDFJ/05Sc8vgJ2i67iJ+6kEeqXboKVai8UQ"
    "f8qz9DJRwU8aKFL8MEAhYkWWwnzFIqkj2vjFMO0cHZUiEhz7PV1R8DYuSDrhusz0KCzcmb8Vz2/4VDAMDx3xayNHKdCD3PWt"
    "DYF9obM7eCqFHNWBOQF7w2dPYXD4ZMTAmJC94cEef4rwtGo7P3vDfa7TDulAn7VAPBnuae5g2urTN0UsjgT4fW5XT1nOw0t4"
    "9azl53MK8EDXfSC4Ae4ZUpfso8lliPpAEQ4UDSb5akFBdh7Xpz3j8b5jo1gtGjzTWZdY/KzxSg4mjo1ZjGUacx91j7pzbTXu"
    "TefXz71CdoAEMxhhb/qVsIEKad8b7q0rRQNZQbmzyZ3FyJzAsp/XaH8Dy2HHiEpHtvVpjUE26HjrdEDdjz2gvmzCsHDWzuSA"
    "TcR3Y0VQu8Jj+vuO6Ke3ohzDidoZ7zsJrxeXYzk5458TEmcOwxxd573DhwEmyB/azBjuJu2K1BrScufDlubyJ35H/OoqqTZe"
    "sXpIH68D3mf27ODgAuU+40cEbeoSzaky3Glpl5+KDte897QPsnZlduZ3b/j9UxxZnWchvtZ8yvMjwz3rKLTc5xzbY0TIGYh3"
    "YlaayjBzOGtCjlaL3oumTPPOpX8ZDZ8ZjU7kuLZzydEDI6cGaAbu4OwZwIJmMu3O4vv7zqo8hS7XydZmPO/Ro15ONzBwxaOQ"
    "Bu7uOiVoalcR6TTGsR9jHvzeymqvsmWdVVGCjuAB8XkWBfW592yG6uIMPTWXZbEo0MKb3qpNSDyLimKZz+94n5X4Jln9Rfxn"
    "cR/MeEnoKywkw6s9+dq3XHoAduVBH1PCE0F9n1pGx6tyv0+sOQUHg+1+ceD8pSeeRpIuzjXuMYyzaDGNIyT4EXMfm6woYBFj"
    "UO3MDO4QUBWEgepoJf727/+nG5ZJowLSN0tyT9Pb/FHwhXU1+a1R2+QXdHjJL1Jt4w92AzS9Ozi9snirxXGFTvCoLMdj46+P"
    "AuVC7cmUnqRFXjwfiWU8fAXc43VJnlt1dBrUWblWMfgFtMo2ytOs0P/LDIf5Y5IszSxNVXLe6OgwFVOgtYA0cU4NRlHG4g0w"
    "uYgSP0EXYvtQWzzXmXs8HyaLZX3rPIqiY4ZWyfrfeiM46vHcF38QT0zIF6C5wOhAs0ipoZS9HeJEe3DG+XPGfNoPbPXlrWVp"
    "IzBY4NqYhFOYe9IfBvhx4OoVfrBwl1bxGD8N5Q9/kzJbOipgVsSYOCcjIWPnwoLxMEjFN4TanGmBppTS3ngMkbuBBgzDHkyB"
    "oSZlluYJvYW2KIfBjHOWkQE48H2n34TacE03IYEf+waT0NpyjmPTZe1Ml3QYNOswRAaPxe2Vg5XtdxlyLPNVX+KOX/HcL0U1"
    "alk77lpw5hL7QG4RMceZktbpDgeWayHo6hhphZs1GB+NB3eikvbQI4GjpEWdaYeRUzP7GiqJFDi+WmKo2OM9vQ06Xf0Trg60"
    "mFWJLU3fNEq+bMiATltUdHCwaA4eAHf4Ba11sPgPtfPaHCsOsLQAsBr4hgZNJiksi9U5nzxQh8DJIbDTxp3LQ9t0hgOoQ1eB"
    "mBlNgShBwSYgdPhaLGGO2DuBY4bZr0AjvdBzXem8Sj/+6jz9tWUxFVqZD2MraokQCFohLMbxdztWMWVPwj82N5LV6ZOTn9LJ"
    "bEKikiDOzFPbSr9Gtd/xgfwC3DDHHxF6QNcfMHK/8tCM5iDFMRWjRyvimxpVYGnG+QRdPKyLkHwO6cwzUIUViucUqzGYNEmJ"
    "Jt1wnmZZHmF7XehqcPQ5sFsMhEsT6UwLMs2G5XawNgZe4wJ97H54gccwH+/5Q2SReBgKRsfDNlj3YU50VynNk51p3K3iMeQd"
    "ZO/MYot4hP08h24yOEvXAvRNWA+Wxv24WQ1gf9qsMJVXrabQWRhj01YBvDEMdTygYFw7908H7JnVijzp3qOrB+oI/3gscEDa"
    "pYzqpfKmWnYfhfpSRrjGO2hYEpbQayApc24twmcOy4kcJtmtJ1Vp+XrUICJf+BMmnP093Y8IdhDqj0lsUgcNSSAeNdg56eOr"
    "J4ljSrVAU1fWlonp9AyxF116ob5hVCzXCki3a3qnR9LyFoPd46ZooDXz2/RaQ0A93lseUHe5q718wFqYpq2kWj5rHtxKlVJR"
    "J1KWBA+B8rt+KL7r3ECTxKlM0IzHsxMASIbttv2I56FDXdM/OzS3+54vwJSQnOuyUicG9MMCX2M4HRWLJYzTNMUTAaK6SBcj"
    "IhnSReT2BWkgaDotokvMqItHJYFSxZs3bwVq05V1dI6TqgkeODyeiooS8osV8VktCoj3hyqO5VcBPjLdjXaAzsofnVaYDjBu"
    "1CbcSIoBxQzA57M2l0mVJJTKpDn0vUxv8CyQW4ciWx60XJ526lwz9U2bI5nblN7JvBJocM60wxfk9dGTPjSWXPggvcu9tJQi"
    "paBqqph8ZWtjkha/ffutLte0j6/WWUFOe9HTDcY+PVNX7Xw9PUZcXOd4+I13Xx+aHoOO1gBzR6OfALERgQEGjWvhx6MPu7zp"
    "h+Dp8GeTEzqaYa5hzpBW2Q4FlXjjAck16KDgpqcip6vZJQwBR8tNs2JKjyhsAMTZ09FkWIHkAGrfxYRM3e0NPPuKAfbGeVP5"
    "EvjnEb1GyfYLMNxxLnWYn0/f/9fjo4/hyauB37uXbIIeMqaehrA/RIS9BmuQde3Eyln1N8s70sYN9sY9tEV68o80yVTvyEBC"
    "JCgTPrECQetJO01Rnk9HIl8OMZNMGd0GLV9SaagtolybdQRZ989/oszibXa+3RksT/MYVKQylEUZdvB2p7qA5Y/5I4CBQt81"
    "1v3+9KWoVkvM+F5R8pk0ygjiks5BL8F4+kGmaAabPgNzFZZlRsk0VDIPVb3VM2izHLT78wIqxz9hvMNFki3leX1M7ynTDVHq"
    "BMwGh8n7o3KnVW7IMqfRbF242O1pUVSAKRk65F9o8v6zkMaECc0YB0rNxJIyY55TDshlak+Oa/Y/D7oTLvdphflpAcZ7EaPj"
    "fZVHV1GaoTAZfFmz3Dnx/uzqYGvT/N8WxpwDXHMs9SThsUusyQtuF3VlCH8Fi5WSVifNWb0fXwrOn0zOlqqQaeoukrJQ6cBi"
    "znIxkznJcgfgVa7cK3GCVwXEgpYREMfB3pPnmHW8jepAbbhGMU76wgKx4FTkDrjaImmz6cnsepy4HNUhpJo5dFrlPpfaGBDw"
    "PBneOaZ6om/ENiCc/eA+ecT9Na0Aa0EmvRxGFQ1KZ6asBNb22WLFos6hLnqRrg6Gs6ua8l+ZDCygb0fv37w/DQHSwY+nh3/x"
    "t9xIuYAZKN8NjJeNOv0tj9FaoWqso45pc9fKeUjsZsTqF3nGWraIjGdX8RsmQqAkmWwlg6nHCLOC8g3ZyVIpq0pULdEZWqKJ"
    "UZF2iNlwg0Yl7aOoFNWIJdtbPZNAMGlLWp8WPJ23TM72J2KXDl46v+5NOuJcG3EJ15whG67zqwMuumOm+mFDBr9jN+iLP4wp"
    "sqjLeVhrQ18H312gS0eSfe3P3pT8NH9MZPyLQQXC+y+YVJQyMyYlmPZkC46x3Mm7j8en4eHp8aHfA1UdtUfgZyPo+gg387UX"
    "o539yVanMnlt1IYkr0fKohWCRf5vwOw82Vlz6ny0TOUXc/AdShQ1oSh+f7gH4+2Zze6KZ09sh6OcL63yC4qcOHAPq4JPVyTg"
    "cyC31VsAPen55XptMujFHNZhr8u3a3WIVCofsxXICc6UOl9lvGpZlWGlwgK6iMpLziuppMltmmQxvM9vSbW5TG6XRYppL3oW"
    "yAXFCul62XUa1xd9S2Z0MOlZwQpQ+0YBcq5aG5AaklBnbXYYHI8e+vjg62MM3tt7Sv8+p0C+A/qXng/o3yf05tke/vsc/gXi"
    "2XNYDgpRymv6jBOb2t14xI27yJOjrfqqy8/99YFIuYkXnUmALsvqLzpT5iZjldxpq3uiRZKcwTfMqQEGQu0GstV7c5IQ6Ijz"
    "xgf4HMrGQKK8jW7eFDOPfiHNf5ToWKs/aPBkOfnxbXh09P749evw3fvTt8evHMPXIRycBvOlWsoSN7/D1C0YxCieOJi3g0mY"
    "Vbvo2dxBle8yCFyu34nTw3cfDo8E588Fre48KRboZJnB1JacCYUFegRGCtjcaKrs8rqYXUT5uZ2cvCinchYAfsi6pperzGjj"
    "fUrBIeZRVePFW2AnZfH4e7MbDQ/BU/dB48kqSnpBW9nTIYdDHOYx+tZWdZe0KK9VD1zNYNGhtz6snjZs4um2gQENFr56LA7f"
    "SeRqcW1oIh/lKOXAvnz9ln8TeSOhhj8dvn178u5H35FAKS0roiCqMbzMc6rsWVj2DEQgLscHXajnRYEKwRlCB71FZlZNS86s"
    "ii3KmAv8QbsqB9R1WX7YCNI/cGzhI/6y336ZuNgWQsSmfVwxT0ZrLv0JeaJZP6DF+PjAOzMI64yGZEiH8k7im8lwyRliOZ0t"
    "dARbmpBjDgWIt7MP/BxsCr/nkApqnnc124yrbBysvjT/+sZDvM+oupTkMU/z+KdiUQCJLi9uPWNAAhNRZnrMAgLx1KXSsGc4"
    "SxMiIxQ12NSwjK6SzPOH1Wrh+T7f9VRdduPO3OeKJDtoWWiaexihBtTw+CkwJZSzQBSqXXqxDy8aAujBc25BJsb6dG/UGzbt"
    "0sJ0CP3nvGw+C/wCIxGraOa+tacbq+n2a1DKPI/lB6OAGcyDO2q3ro/v5Ll+iR9PDT/LizNyOqhbUJoqBEFpszPrQMsX20V3"
    "Qr6S/jTBD3HToHNldhW6vTUbuQYf0iq/ppUWUjjB4Iu9NRXSRmuo4h6UYNGTzH/zzam1KbI2yH11j60q08N5GMdNUE3j7ZSR"
    "ObhvVaaYQarxdzY57G9U5l2ZNmroDJbR+7cmZAZamqa5vVcrhzZQg7g+eKHZZEZ2gqxCAfXb3En911ZMOtEzrkFHFRm/ybYw"
    "/Fs2siZSwu7GumxDzg2W9dmGOug6nKly4+0Nbai0QVP6QIjqMkUv9KiTzBMMbE5Mjfd9ooN76OZxvf28zw2LdmYk8rBq5+p6"
    "05ybJdZ4WO2id95gZ1fQPY3fPwPLjxILfp2f0fAAmn7GDr4b8sh7zfkql9sKSWzP/6az7VrMzeqwAtTU+zMXA59Y5nhPYcnS"
    "KboWiDOkVMJJ3DIh+UJHQw+naI/6ST/zeB2b0C8VaflWe7+O1CIkrKAB+kjX2jDeYXnBOoc6lvlBvVN1/XukmHQcZpZ7YsQh"
    "R3yvailjWDk4D48bckbIkeP2Y02CUN1Nc3g1XVuXxZUPpDqTU6qHwM0Dg/Zcl57dzLUJqpS6zRrr0l8gLHe8xL93u5WGOpC7"
    "Z1asw1yuP+akOkWNmD5RgtKpAEmteCzADvYVnyVpfGkktSIvLy/aEBFfX6pUsTu/nhbnMlYUx5dYSqLousFqeWtGiQY/3kWk"
    "4xtoi8C1TU4mzVgtxmEKA35GEEcM9zutftfsWy2buEHXYeZHZzq5E5G05C0T5jdfsAvUFh7GKjHazfMdp1K1SjJMOVdo9DnB"
    "JHfBrsmafXyNntxc74GwNTboAq4+6+e0f1ZXMvRF19B2LxIFZyoF4i1v+X4dvAKNKVpphPopa9wNVhRKO8eB3CzGG5p/4YwD"
    "dGVF6zSiXFkMunW3o/RjYjP0OF/un1jK3b7fL1g4ukibAL2979jgfHSHOAJJTGm2ySrz14o5M9TYgcKG8cZ3RPoxYBny1CDf"
    "9mKjCD8Lmgxg2hDa73qhSabYohhoDXQsLRlInRXXmqHVsj/PiO/qM6LWGVjd8xu/le3FOZ7MIyAPtsp+5nOLHITB9rrMgcCS"
    "pI0/wcipSDsWoQ5X0C15WuTJny+KTMWBtJmtKZMCVi1LvqmMt1QxlmRXXffKddKcbvuWF+C184IRBrRXgjeVVhdFTVdgUi5/"
    "rMZxChUjCGwkkMdaOBwR/X1lOl3VjLrjfEmAe1MVWqa81byrPNq76MBWw8DQ5GHPSh81jYD4gvP2cAmHQDKDYvaEgS501yfe"
    "JgBw6ACMFny8/mDIhrYuejJBl+FwO2lPrrM1TaXUvOpvMuo/UvYfZGr24/APM/A/wgzszoeZHl1zmABZdnwlFr1BLw/MUy7y"
    "6M+6Sr4h7Uli4gr4yiOXxODoYAZDbY8G7Aeya7u7YH35xtLrnJaWiu0lGakU8Wqb0diMcvTKkgbqNkQddZTboARFwAYquk1n"
    "PMgiPCXgxvaxfoDBPGWyydkSqd+A0JZ/HnCMRO4YVPJWK3/iOrvZnmbpOb9pjlpb3Nnr53qv5UYPd8AXL8S+CV0XGbRAYGlV"
    "y2jWGCaq5g5umeSrxRjJG5t8pjX5zDcK+75S8sBo8++ef7JiNEwmFlmG8dxwZTKQb3nKhEeS+mNeCymbX+OPdLGCr/CBbH4R"
    "xobuj384NbqCodH3/v5+jcbQc7g22puX6e4J5vsjIsLGsaEockPPBmp68qXuzqB2aH9AEbglDrhAzzFW06HoLHqnO/G3dqgw"
    "Vn7Q70dc417hyi7vSi+0yTrPR0NveMfCb+R5WRMZ5/J3dKZU4XgPd8pdwDUi6IHe9adw1biZhLO1lPhi7NaV2muyJxPHcW7Z"
    "yGZc3cKsNyeFlrzBsLNR4FR0apHzJMg0UoY8CrTT3jFdggX8tBWj9MMw0HWgdL7oa4G24676+T+bDluyUwcwTCvQB9pu/rXF"
    "zp989ZAnN8sI2Zhyp2i7mRIeV/htTpzarRu/e5xTGnVu5JxyarObigZloscsGuSQ2JLBsNIFHvnCs2LS/yFzNjZJJLsXP8tp"
    "0fvuOnyaZQsYfCURYWGo3DpyQ/vvu8XvOH76/OGXX5HCo6NjHr5b5eTZ1dOMcIIOYtTKgcsJhH+QHpJfVwnMH+Xy5PMkUQ2T"
    "ghOg+ULaSbQ14Htu5uNt6zqsjRMRaKei8vNMXlFNp+uY4aiZ51nhQgO/ezIpo4AUT4fivq/bupHaoCrP7nRgUEOgz/pYe3Zf"
    "SjV2O2fWO6LueVPVXXdgu3ZNpTmHC99omcYbx0O3GzT/gvVlokPv31jtb6atY7WifTAaMfJSmORWF3SZogfzCoJzjCmN6P4q"
    "/+9g9kjHuBp+DNT3zAO15jFF0xx5iF/PODrbc1Om6wSt9l67SrXn2syywFO70gMX0Zx7ygTRL9McEIa7y/x84LYFefYBD87u"
    "3k3EtM4U1JcWjuzIHNW150E3tw4N3qDzEEbZZgONBTdOHGiqGSvrsYm73zXJXussnRqRSeda8ralrdLN0eUzF5/bxfJFoG2j"
    "mWjyXJ684penztx0lqJzk9yZ38acMgdkY+vYGse++3Ed967dsSnejvKm++JtzpfNd8VVyF1yU1MqmR4DzVoo7cKwFwHba3wU"
    "X5HDxPe37HlthHorcOWXlgTOV1EZl1GaaUI3yQE+tIUntMI5apFSg7wCbov7wdXAEv1S/DbAfD55TJsPCpWe1FYtok1tr33d"
    "0RY1ePdzSLvduutyNWk8jLXTENOVocSgVAY0H5WGq6lRrtfpjgkgp1TAacfj5MpokDMLajVf2Lmks+BNkzjlWA+XIod/UlS8"
    "nVjBHnh0EJu5hWl+UmNInS71zshvmScdpE0nP55p4Iia5M0W4Qw36co06uZvsg1oxC5pbpOkGWaSp0dEWTZ8ZnZoQhZZHnn+"
    "kJN5cmq2FsjEVhy5pU27++D7Tu99d8T9743Y8M6IDe+LcN4Vcdc9Ee0VlL2B+O2ll+6K9mWY1jVXfWA3vDNz65tcWeG8ruLw"
    "3V/0y3C2Nry3ou137w0WztsrmG77bsdu3Q/TW4Xg5ybFv7lkfErwS6srbJx/cnlp7r8vHafFpgm3TcaF0ExSb7+HdFfEWLAm"
    "37z23anTWRib/SSGgx21gHY63dGlW7Ayve1oa6PDk8wldFwaW8R3FBxyZJqNn6uo7kXi4w9tBnijPWKyzTc3KK09NX8EBdmv"
    "mYhRfm020AhAC3SVfyXP/53i+WqYHLYaOTqaPH6PFE6BeNQ0PzEL22pOC4EkgAaQVEQHb1ca+fuWXKndimI/SSeXTSuFXNKL"
    "umbsb//rf6u80owulWuOcFtCvHI4wghLl+trvsqyEFROsA6h7kW4TJcJ+tqkrmycb/lmUVKd29ntYCkowIrVJi4s/Wr244Nj"
    "7FaelCwS9VxRMpKHIpQuwGBrjq3qAunv4jCR8fj/mh8MxSl7CqRz1Lxhnpv827//XyOVIKGhbmhX8hbnkmtJRL3PX2zbcqOI"
    "HhP+fdxsdt7RtowJ0zEOj4fiDWZyziirkZYGD4OaNbiYUVimOMaMNKDIcnhZsvTNUepmve7Lmq7ZaY7azfhuHp7oaH6TidCT"
    "RjtywUu9gQHLHxLB/qzWa1BoG1ubX38++BfNgSDIx8Jp9Ft8fDBRtVfYpMXHtBm0U+dT0KLLLSHDVWCaO8nzh4P7DZCcwfsc"
    "5VsHcZPZdGbz767YzTcevhah8QOx3MA87bOjf3MjOq+UzrPlvgiF7CfubuNUHIyE462p7g5cHsoBBTN0Xls1p6s0i0POJ0u2"
    "c0jRWiDWnB+s2hSNGxppSqGq421g9zOnVO6hTgghGvDU3b6PFpSODMFTyPY7q04PW4WaPV8cba7N/SpxWFvGAfOulSXB3lXM"
    "AflujUnCvrtgYCXiOL7BeNSYg5bRwMGtORDKCWgVl9volJ3TPiisKeSEeIQClMAyvTEzbwwcIT8hgAvbeoCiKy7IAiN3xnTF"
    "xwGHP9h1XULJUdlVrAXFpiEzGKksnORVDYoSHU5EjWl3CX/5YgEM1wZCxdxir5TSVwLdpHnVerQof6ipAJJaQTdugQbx/wDa"
    "GJtu"
)).decode("utf-8")
_optimizer_namespace = {}
exec(compile(_optimizer_source, "<recall_precision_upgrade>", "exec"), _optimizer_namespace)
_optimizer_namespace["install_recall_precision_overrides"](globals())


In [ ]:
# 6B. App Engine 3 GB Runtime Guardrails — runs immediately after the recall/precision upgrade
# This is embedded code, not a separate service or a database change.
if "_optimizer_source" not in globals():
    raise RuntimeError("Run the Recall & Precision Upgrade cell before these App Engine guardrails.")

_app_engine_source = _optimizer_source

def _replace_once(source, old, new, label):
    if source.count(old) != 1:
        raise RuntimeError(f"Could not apply App Engine guardrail: {label}")
    return source.replace(old, new, 1)

_app_engine_source = _replace_once(
    _app_engine_source,
    '''            with Image.open(io.BytesIO(image_bytes)) as source:
                source = ImageOps.exif_transpose(source)
''',
    '''            with Image.open(io.BytesIO(image_bytes)) as source:
                # Do this header-only dimension check before decoding a source image.
                # 64 MP bounds even a worst-case RGBA decode on a 3 GB instance.
                if source.width * source.height > 64_000_000:
                    raise ValueError("Image exceeds the 64 MP App Engine processing safety limit")
                source = ImageOps.exif_transpose(source)
''',
    "maximum final-audit decoded pixels",
)

_app_engine_source = _replace_once(
    _app_engine_source,
    '''        except Exception as exc:
            print(f"Warning: failed to normalize image for visual inference: {exc}")
            return image_bytes
''',
    '''        except ValueError as exc:
            # Never fall back to transmitting an unsafe raw image after the pixel cap.
            if "App Engine processing safety limit" in str(exc):
                raise
            print(f"Warning: failed to normalize image for visual inference: {exc}")
            return image_bytes
        except Exception as exc:
            print(f"Warning: failed to normalize image for visual inference: {exc}")
            return image_bytes
''',
    "safe oversized-image failure",
)

_app_engine_source = _replace_once(
    _app_engine_source,
    '''        max_side = 1600
                if max(source.size) > max_side:
                    resampling = getattr(Image, "Resampling", Image).LANCZOS
                    source.thumbnail((max_side, max_side), resampling)
''',
    '''        max_side = 1600
                # JPEG draft mode avoids materialising a large source before resizing.
                try:
                    source.draft("RGB", (max_side, max_side))
                except Exception:
                    pass
                if max(source.size) > max_side:
                    resampling = getattr(Image, "Resampling", Image).LANCZOS
                    source.thumbnail((max_side, max_side), resampling)
''',
    "bounded final-audit image decode",
)
_app_engine_source = _replace_once(
    _app_engine_source,
    '''                storage_client = ns["storage"].Client(project=ns.get("PROJECT_ID"))
                return storage_client.bucket(bucket_name).blob(blob_name).download_as_bytes()
''',
    '''                storage_client = ns.get("_audit_storage_client")
                if storage_client is None:
                    storage_client = ns["storage"].Client(project=ns.get("PROJECT_ID"))
                    ns["_audit_storage_client"] = storage_client
                return storage_client.bucket(bucket_name).blob(blob_name).download_as_bytes()
''',
    "reusable GCS client",
)
_app_engine_source = _replace_once(
    _app_engine_source,
    '''        try:
            import cv2

            with Image.open(io.BytesIO(candidate_bytes)) as candidate_source:
                candidate_source = ImageOps.exif_transpose(candidate_source)
                # Downsample before RGB conversion so large hero images do not create an
                # unbounded decoded array.  2048 px preserves substantially more detail
                # for a small embedded target while keeping four active workers safe.
                candidate_source.thumbnail((2048, 2048), getattr(Image, "Resampling", Image).LANCZOS)
''',
    '''        try:
            import cv2
            # OpenCV stays CPU-only and cannot create per-worker native thread pools.
            try:
                cv2.setNumThreads(1)
            except Exception:
                pass

            with Image.open(io.BytesIO(candidate_bytes)) as candidate_source:
                if candidate_source.width * candidate_source.height > 64_000_000:
                    raise ValueError("Candidate exceeds the 64 MP App Engine processing safety limit")
                candidate_source = ImageOps.exif_transpose(candidate_source)
                # Preserve small embedded targets while bounding App Engine image work.
                try:
                    candidate_source.draft("RGB", (2048, 2048))
                except Exception:
                    pass
                candidate_source.thumbnail((2048, 2048), getattr(Image, "Resampling", Image).LANCZOS)
''',
    "bounded local OpenCV decode",
)
_app_engine_source = _replace_once(
    _app_engine_source,
    '''        policy = audit_config.setdefault("_search_policy", _policy(audit_config, bool(reference_image_path)))
        workers = max(1, int(audit_config.get("llm_concurrency", policy["llm_concurrency"])))
        batch_size = max(workers, int(audit_config.get("llm_batch_size", policy["llm_batch_size"])))
''',
    '''        policy = audit_config.setdefault("_search_policy", _policy(audit_config, bool(reference_image_path)))
        # Use the policy caps; direct runtime parameters may not bypass 3 GB safety.
        workers = policy["llm_concurrency"]
        batch_size = policy["llm_batch_size"]
''',
    "final-audit worker cap",
)

_policy_start = _app_engine_source.index("    def _policy(")
_policy_end = _app_engine_source.index("    def build_fused_query_text", _policy_start)
_policy_prefix = _app_engine_source[_policy_start:_policy_end]
_policy_replacement = _policy_prefix.replace(
    '        color_is_discriminating = exact_mode or version_mode or any(term in text for term in color_terms)\n        return {\n',
    '''        color_is_discriminating = exact_mode or version_mode or any(term in text for term in color_terms)
        # Default to the declared App Engine 3 GB memory profile.  These clamp active
        # image payloads; they do not lower retrieval recall or change the database.
        runtime_profile = str(audit_config.get("runtime_profile", "app_engine_3gb")).strip().lower()
        constrained_runtime = runtime_profile in {"app_engine_3gb", "memory_constrained", "3gb"}
        llm_worker_cap = 2 if constrained_runtime else 4
        local_worker_cap = 2 if constrained_runtime else 4
        llm_workers = min(llm_worker_cap, max(1, int(audit_config.get("llm_concurrency", 2 if constrained_runtime else 4))))
        local_workers = min(local_worker_cap, max(1, int(audit_config.get("local_verification_concurrency", 2 if constrained_runtime else 4))))
        llm_batch_cap = 4 if constrained_runtime else 8
        llm_batch_size = min(llm_batch_cap, max(llm_workers, int(audit_config.get("llm_batch_size", llm_workers))))
        return {
''',
    1,
)
if _policy_replacement == _policy_prefix:
    raise RuntimeError("Could not apply App Engine guardrail: policy setup")
_policy_replacement = _policy_replacement.replace(
    '            "color_is_discriminating": color_is_discriminating,\n',
    '            "color_is_discriminating": color_is_discriminating,\n            "runtime_profile": runtime_profile,\n',
    1,
)
_policy_replacement = _policy_replacement.replace(
    '            "llm_concurrency": max(1, int(audit_config.get("llm_concurrency", 8))),\n            "llm_batch_size": max(1, int(audit_config.get("llm_batch_size", 8))),\n',
    '            "llm_concurrency": llm_workers,\n            "llm_batch_size": llm_batch_size,\n',
    1,
)
_policy_replacement = _policy_replacement.replace(
    '            "local_verification_concurrency": max(1, int(audit_config.get("local_verification_concurrency", 4))),\n',
    '            "local_verification_concurrency": local_workers,\n',
    1,
)
if ("\"llm_concurrency\": llm_workers" not in _policy_replacement or
        "\"llm_batch_size\": llm_batch_size" not in _policy_replacement or
        "\"local_verification_concurrency\": local_workers" not in _policy_replacement):
    raise RuntimeError("Could not apply App Engine guardrail: policy limits")
_app_engine_source = _app_engine_source[:_policy_start] + _policy_replacement + _app_engine_source[_policy_end:]

_app_engine_namespace = {}
exec(compile(_app_engine_source, "<app_engine_3gb_guardrails>", "exec"), _app_engine_namespace)
_app_engine_namespace["install_recall_precision_overrides"](globals())
print("App Engine 3 GB guardrails active: CPU-only OpenCV; max 2 live visual tasks and 2 local image workers.")


## Offline Validation Matrix — No AlloyDB or Gemini Calls

Run the **Recall & Precision Upgrade** cell immediately above first. This local matrix creates synthetic fixtures and validates: exact/reference image matching, resized copies, embedded targets, crops, blur, transparency processing, solid-versus-gradient policy handling, text-only routing, the disabled semantic LLM reranker, and bounded final-audit concurrency.

It is deliberately read-only: it does not connect to AlloyDB, download GCS objects, call Gemini, or write audit results. It validates functional behavior and resource controls; production recall and precision still need the separate labeled, read-only evaluation described after the matrix.

In [ ]:
# 6A. Offline validation matrix — runs locally only (no AlloyDB, GCS, or Gemini calls)
# Run the "Recall & Precision Upgrade" cell first.
import asyncio
import io
import importlib.util
import time

import numpy as np
import pandas as pd
from PIL import Image, ImageDraw, ImageFilter

required_helpers = (
    "_local_visual_score_for_validation",
    "_audit_search_policy_for_validation",
    "_visual_candidate_set_for_validation",
    "process_transparency",
    "build_fused_query_text",
    "run_semantic_reranking_and_filter",
    "run_llm_inference_on_dropoff_results",
)
missing_helpers = [name for name in required_helpers if name not in globals()]
if missing_helpers:
    raise RuntimeError(
        "Run the Recall & Precision Upgrade cell first. Missing helpers: " + ", ".join(missing_helpers)
    )

validation_rows = []

def add_result(use_case, status, observation):
    validation_rows.append(
        {"Use case": use_case, "Status": status, "Observation": observation}
    )

def png_bytes(image):
    buffer = io.BytesIO()
    image.save(buffer, format="PNG")
    return buffer.getvalue()

# A deliberately feature-rich synthetic reference mark. It supports both template and
# keypoint verification, without using any real user image or database object.
reference = Image.new("RGB", (360, 240), "white")
draw = ImageDraw.Draw(reference)
draw.rounded_rectangle((14, 14, 346, 226), radius=22, outline="#1A73E8", width=10)
draw.ellipse((52, 58, 172, 178), fill="#EA4335", outline="#202124", width=5)
draw.polygon([(220, 58), (312, 118), (220, 178)], fill="#34A853", outline="#202124")
draw.line((192, 54, 192, 184), fill="#FBBC04", width=9)
draw.text((104, 191), "AUDIT", fill="#202124")
reference_bytes = png_bytes(reference)

# Visual transformations that represent the retrieval/audit cases we care about.
resized = reference.resize((540, 360))
embedded = Image.new("RGB", (960, 640), "#F1F3F4")
embedded.paste(reference.resize((151, 101)), (745, 482))
cropped = reference.crop((24, 18, 334, 222))
blurred = reference.filter(ImageFilter.GaussianBlur(radius=2.8))
unrelated = Image.new("RGB", (720, 480), "#202124")
unrelated_draw = ImageDraw.Draw(unrelated)
unrelated_draw.rectangle((70, 70, 650, 410), fill="#8AB4F8")
unrelated_draw.ellipse((280, 150, 470, 340), fill="#FBBC04")

if importlib.util.find_spec("cv2") is None:
    for label in ("Exact image-to-image", "Resized near-duplicate", "Small embedded target", "Cropped target", "Blurred target", "Unrelated negative"):
        add_result(label, "SKIPPED", "OpenCV is unavailable. Run the installation cell, then rerun this matrix.")
else:
    reference_rgb = np.asarray(reference.convert("RGB"))
    visual_cases = [
        ("Exact image-to-image", reference_bytes, 0.90),
        ("Resized near-duplicate", png_bytes(resized), 0.70),
        ("Small embedded target", png_bytes(embedded), 0.45),
        ("Cropped target", png_bytes(cropped), 0.45),
        ("Blurred target", png_bytes(blurred), 0.45),
        ("Unrelated negative", png_bytes(unrelated), None),
    ]
    visual_evidence = {}
    for label, candidate_bytes, threshold in visual_cases:
        evidence = _local_visual_score_for_validation(reference_rgb, candidate_bytes)
        visual_evidence[label] = evidence
        score = float(evidence["local_visual_score"])
        method = evidence["local_visual_method"]
        if threshold is None:
            exact_score = float(visual_evidence["Exact image-to-image"]["local_visual_score"])
            status = "PASS" if score < exact_score else "CHECK"
            add_result(label, status, f"score={score:.3f}; method={method}; exact baseline={exact_score:.3f}")
        else:
            status = "PASS" if score >= threshold else "CHECK"
            add_result(label, status, f"score={score:.3f}; method={method}; expected local signal ≥ {threshold:.2f}")

# Verify lossless PNG output, alpha compositing, and active-payload memory capping.
transparent_large = Image.new("RGBA", (2300, 1800), (0, 0, 0, 0))
transparent_draw = ImageDraw.Draw(transparent_large)
transparent_draw.ellipse((300, 250, 2000, 1550), fill=(255, 255, 255, 190))
processed = process_transparency(png_bytes(transparent_large))
with Image.open(io.BytesIO(processed)) as processed_image:
    output_is_png = processed_image.format == "PNG"
    output_is_bounded = max(processed_image.size) <= 1600
add_result(
    "Transparent / oversized asset",
    "PASS" if output_is_png and output_is_bounded else "FAIL",
    f"normalized to PNG; max side ≤ 1600 px = {output_is_bounded}",
)

# Validate goal routing without asking Gemini to create the configuration.
policy_for_validation = _audit_search_policy_for_validation
exact_policy = policy_for_validation(
    {"audit_goal": "Find the exact uploaded profile picture everywhere it occurs."}, True
)
legacy_policy = policy_for_validation(
    {"audit_goal": "Find the old solid brand mark and reject the newer gradient version."}, True
)
text_only_policy = policy_for_validation(
    {"audit_goal": "Find product screenshots containing checkout buttons."}, False
)
policy_ok = (
    exact_policy["mode"] == "exact_or_near_duplicate"
    and exact_policy["local_verification_budget"] > 0
    and legacy_policy["mode"] == "version_sensitive"
    and legacy_policy["color_is_discriminating"]
    and text_only_policy["mode"] == "semantic_visual"
    and text_only_policy["local_verification_budget"] == 0
)
add_result(
    "Exact, legacy/color, and text-only routing",
    "PASS" if policy_ok else "FAIL",
    f"exact={exact_policy['mode']}; legacy={legacy_policy['mode']}; text-only={text_only_policy['mode']}",
)

# The App Engine profile must clamp caller-supplied concurrency rather than trusting it.
overloaded_policy = policy_for_validation(
    {
        "audit_goal": "Find the exact uploaded profile picture everywhere it occurs.",
        "llm_concurrency": 32,
        "llm_batch_size": 32,
        "local_verification_concurrency": 32,
    },
    True,
)
runtime_guardrail_ok = (
    exact_policy.get("runtime_profile") == "app_engine_3gb"
    and overloaded_policy["llm_concurrency"] == 2
    and overloaded_policy["llm_batch_size"] == 4
    and overloaded_policy["local_verification_concurrency"] == 2
)
add_result(
    "3 GB App Engine live-work cap",
    "PASS" if runtime_guardrail_ok else "FAIL",
    f"LLM workers={overloaded_policy['llm_concurrency']}; batch={overloaded_policy['llm_batch_size']}; local workers={overloaded_policy['local_verification_concurrency']}",
)

legacy_query = build_fused_query_text(
    {
        "audit_goal": "Find the old solid brand mark and reject the newer gradient version.",
        "inclusion_criteria": ["Use the old solid color treatment."],
        "_search_policy": legacy_policy,
    }
)
add_result(
    "Solid-versus-gradient preservation",
    "PASS" if "solid-versus-gradient" in legacy_query else "FAIL",
    "The fused retrieval query preserves color/style as match evidence rather than globally ignoring it.",
)

# Quick Mode must retain its shape: 30 High + 30 Borderline, never a pooled top 60.
quick_high_input = pd.DataFrame(
    [{"asset_id": f"high-{index}", "canonical_key": f"high-{index}", "relevance_score": 100 - index, "segmentation_band": "high"} for index in range(35)]
)
quick_edge_input = pd.DataFrame(
    [{"asset_id": f"edge-{index}", "canonical_key": f"edge-{index}", "relevance_score": 50 - index, "segmentation_band": "borderline"} for index in range(35)]
)
quick_high, quick_edge = _visual_candidate_set_for_validation(
    quick_high_input, quick_edge_input, pd.DataFrame(), {"_search_policy": exact_policy}, quick_mode=True
)
quick_quota_ok = (
    len(quick_high) == 30
    and len(quick_edge) == 30
    and quick_high.iloc[0]["asset_id"] == "high-0"
    and quick_edge.iloc[0]["asset_id"] == "edge-0"
)
add_result(
    "Quick Mode 30 High + 30 Borderline quota",
    "PASS" if quick_quota_ok else "FAIL",
    f"selected High={len(quick_high)} and Borderline={len(quick_edge)}; bands are not pooled.",
)

# This confirms the legacy description-only LLM reranker is bypassed before final vision audit.
rerank_high = pd.DataFrame([{"asset_id": "a", "relevance_score": 0.5}, {"asset_id": "b", "relevance_score": 0.9}])
rerank_edge = pd.DataFrame([{"asset_id": "c", "relevance_score": 0.4}])
reranked_high, reranked_edge, reranked_low = await run_semantic_reranking_and_filter(
    rerank_high, rerank_edge, {"audit_goal": "offline validation"}
)
rerank_ok = (
    len(reranked_high) == 2
    and len(reranked_edge) == 1
    and reranked_low.empty
    and reranked_high.iloc[0]["asset_id"] == "b"
)
add_result(
    "Description-only LLM reranker bypass",
    "PASS" if rerank_ok else "FAIL",
    "Candidate ranks are retained deterministically; no remote cross-encoder is invoked.",
)

# Exercise final-audit batching using a local coroutine. It deliberately replaces the
# final vision call only for this test and restores the original function immediately.
concurrency = {"active": 0, "peak": 0}
original_single_audit = globals().get("run_llm_audit_single")

async def _validation_single_audit(asset, audit_config, _executor=None, reference_image_part=None):
    concurrency["active"] += 1
    concurrency["peak"] = max(concurrency["peak"], concurrency["active"])
    await asyncio.sleep(0.01)
    concurrency["active"] -= 1
    return {
        **asset,
        "matches_criteria": False,
        "match_confidence": 0,
        "match_rationale": "Local bounded-concurrency validation only.",
    }

globals()["run_llm_audit_single"] = _validation_single_audit
try:
    bounded_candidates = pd.DataFrame(
        [{"asset_id": f"fixture-{index}", "relevance_score": 1.0 - index / 20} for index in range(11)]
    )
    bounded_results = await run_llm_inference_on_dropoff_results(
        bounded_candidates,
        pd.DataFrame(),
        {"llm_concurrency": 3, "llm_batch_size": 4},
        reference_image_path=None,
    )
    bounded_ok = len(bounded_results) == 11 and concurrency["peak"] <= 3
    add_result(
        "Bounded final visual-audit work",
        "PASS" if bounded_ok else "FAIL",
        f"audited={len(bounded_results)} fixtures; observed concurrent tasks={concurrency['peak']} (cap=3)",
    )
finally:
    if original_single_audit is None:
        globals().pop("run_llm_audit_single", None)
    else:
        globals()["run_llm_audit_single"] = original_single_audit

validation_df = pd.DataFrame(validation_rows)
print("Offline validation completed. No AlloyDB, GCS, Gemini, or database-write calls were made.")
try:
    from IPython.display import display
    display(validation_df)
except Exception:
    print(validation_df.to_string(index=False))


In [ ]:
# TEST RUN (Stage 1): Generate Audit Rules & Configuration
# Run this cell to upload your reference image and generate the rule configuration.
import nest_asyncio
nest_asyncio.apply()

# Dynamic Reference Image Detector (Triggered via Colab Interactive Upload)
reference_image_path = None
try:
    from google.colab import files
    print("[OPTIONAL] Upload a reference image for comparative compliance audit:")
    uploaded = files.upload()
    if uploaded:
        reference_image_path = list(uploaded.keys())[0]
        print(f" Reference image uploaded: {reference_image_path}")
    else:
        print(" No reference image uploaded. Running text-only audit goal.")
except Exception as e:
    reference_image_path = None

# Enterprise Audit Goal (Modify this as needed)
TEST_AUDIT_GOAL = "Find all pages with this exact image"

# Step 1: Generate configuration rules
audit_config_global = None
try:
    audit_config_global = await generate_audit_config_only(
        user_goal=TEST_AUDIT_GOAL,
        reference_image_path=reference_image_path
    )
    print("💡 TIP: You can inspect and tweak 'audit_config_global' directly in the cell below before running Stage 2.")
except Exception as e:
    print(f"Error generating audit configuration: {e}")


In [ ]:
# TEST RUN (Stage 2): Execute Visual Search & Parallel Audit
# Run this cell to execute retrieval, segmentation, and LLM inference.
# You can uncomment and modify rules below to calibrate config before executing.

if 'audit_config_global' in globals() and audit_config_global is not None:
    # OPTIONAL CALIBRATION TUNING:
    # If the generated AI rules were slightly off, you can uncomment and edit them here:
    # audit_config_global["inclusion_criteria"] = [
    #     "The image must contain the legacy Google Pay logo featuring the interlocking loops design."
    # ]
    # audit_config_global["exclusion_criteria"] = [
    #     "Exclude images containing the current compliant Google Pay GPay wordmark button logo."
    # ]

    df_results_global = None
    try:
        df_results_global = await run_full_test_bench_pipeline_execution(
            audit_config=audit_config_global,
            reference_image_path=reference_image_path
        )
    except Exception as e:
        print(f"Error executing test bench pipeline: {e}")
else:
    print(" Please run Stage 1 cell first to generate 'audit_config_global'.")
